# Kawasaki disease follow-up echocardiography — JAHA figures and tables (local, one click; v3)

Reproduces every figure and table of the JAHA submission: main Figures 1–5 (composite, multipanel) and Tables 1–2, the supplemental
figure and the supplemental tables (files are named by content; the S numbers are assigned when the supplement is assembled), and the
figure legends and table notes (Wuhan Children's Hospital; derivation 2018–2024 without 2021, temporal validation from March 8, 2025).

**How to run.** Keep this notebook in the package folder next to its `inputs/` subfolder (layout in `README_how_to_run.txt`),
open it in Jupyter or VS Code and Run All. Outputs go to `output_kd_jaha_figures_tables_v3/` next to the notebook. The first code
cell checks that every input is present and unchanged (SHA-256 against `INPUT_MANIFEST_sha256.csv`) and stops with a list of the
missing or changed files otherwise.

**Rules.** No new outcome analysis. Every number is read from the locked results (`inputs/locked/`) or recomputed from the frozen
research-ID tables (`inputs/frozen_patient_level/`) and checked against the locks; the run stops if any check fails. Table 1,
Figure 1B, Figure 3C and the validation curve of Figure 2C are descriptive additions computed from the same frozen data.

**Style.** Arial (Liberation Sans, which has identical metrics, if Arial is not installed); short bold axis titles; white background,
no gridlines; panel letters upper left, aligned with the y-axis title; one color scheme for all figures, the NEJM journal palette
(derivation blue #0072B5, validation red #BC3C29); data labels kept; statistics, reference lines and abbreviations in the figure
legends; no figure titles. Width at most 7.0 in (178 mm), height at most 9.0 in, fonts 8 pt or larger; PDF (vector), TIFF (600 dpi,
LZW) and PNG (600 dpi). Before each figure is saved, an automatic check stops the run if two texts overlap or touch, if a legend lies
on data or text, or if anything falls outside the figure (section 2).

In [ ]:
import os, re, json, glob, hashlib, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.patches import FancyBboxPatch, Rectangle
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)

BASE = Path.cwd()
IN = BASE / 'inputs'
V4, V41, V5 = IN / 'locked' / 'v4', IN / 'locked' / 'v4_1', IN / 'locked' / 'v5'
COR = IN / 'locked' / 'coronary_v1'
FROZEN = IN / 'frozen_patient_level'
KIT = ADD = FROZEN
OUT = BASE / 'output_kd_jaha_figures_tables_v3'

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

MANIFEST = BASE / 'INPUT_MANIFEST_sha256.csv'
if not MANIFEST.exists():
    raise FileNotFoundError(f'INPUT_MANIFEST_sha256.csv not found in {BASE}. Open the notebook from the package folder '
                            '(Jupyter/VS Code must run it with the notebook folder as the working directory).')
man_in = pd.read_csv(MANIFEST)
missing = [p for p in man_in['file'] if not (BASE / p).exists()]
if missing:
    raise FileNotFoundError('Missing input files (see README_how_to_run.txt):\n  ' + '\n  '.join(missing))
changed = [p for p, h in zip(man_in['file'], man_in['sha256']) if sha256(BASE / p) != h]
if changed:
    raise ValueError('These input files differ from the frozen versions:\n  ' + '\n  '.join(changed))
OUT.mkdir(exist_ok=True)
print(f'{len(man_in)} input files present and unchanged; outputs go to {OUT.name}/')

LOCKS = {V4 / 'Results_locked_v4.txt': '5940f6a3c0bb69cc3d870fd48b60beaf2ca593911657c7b019ed0f4bb34d5aa3',
         V41 / 'Results_locked_v4_1.txt': '5600d5765bb34348dcfeab220325a4a7abe1e8b28a979d1ee6ac18777d14ee55',
         V5 / 'Results_locked_v5.txt': 'e4b6dbb9cf3a204c817f9fea592da87f56c50539335833b6c61c4116f25b2553'}
for p, h in LOCKS.items():
    got = sha256(p)
    print(f'{p.name}: {got[:16]}…', 'OK' if got == h else 'MISMATCH')
    assert got == h, f'{p} does not match its locked SHA-256'
LOCK_TEXT = '\n'.join(open(p, encoding='utf-8').read() for p in LOCKS)

def in_lock(s):
    """True if the exact string s (for example '26/1,618' or '1.61%') occurs in one of the locked Results files."""
    return re.search(r'(?<![\d.])(?<!\d,)' + re.escape(s) + r'(?!\d|,\d)', LOCK_TEXT) is not None

def assert_lock(*strings):
    for s in strings:
        assert in_lock(s), f'"{s}" not found in the locked Results files'

def wilson(k, n, z=1.959964):
    """Wilson score 95% CI in percent."""
    if n == 0:
        return (np.nan, np.nan)
    p = k / n; d = 1 + z ** 2 / n; c = p + z ** 2 / (2 * n); h = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2))
    return (max((c - h) / d * 100, 0.0), min((c + h) / d * 100, 100.0))

def nfmt(x):
    return f'{int(x):,}'

# locked tables and key numbers (aggregate values only; key_numbers.json also holds report excerpts, which are never printed)
T2 = pd.read_csv(V41 / 'Table2_sequential_yield.csv')
T3A = pd.read_csv(V41 / 'Table3a_first_documentation_by_window.csv')
T4A = pd.read_csv(V41 / 'Table4a_stratified_yield.csv')
T4B = pd.read_csv(V41 / 'Table4b_firth_logistic.csv')
M3B = pd.read_csv(V41 / 'TableM3b_strategies.csv')
KN4 = json.load(open(V41 / 'key_numbers_clean.json', encoding='utf-8'))
KN41 = json.load(open(V41 / 'key_numbers_v4_1_modules.json', encoding='utf-8'))
KN5 = json.load(open(V5 / 'key_numbers_v5.json', encoding='utf-8'))
V51 = pd.read_csv(V5 / 'TableV5_1_validation_vs_v4.csv').set_index('key')
V54 = pd.read_csv(V5 / 'TableV5_4_P1_events.csv')
print('locked tables loaded')

## Figure style

In [ ]:
have_arial = any('arial' in f.name.lower() for f in font_manager.fontManager.ttflist)
FONT = 'Arial' if have_arial else 'Liberation Sans'
print('font used:', FONT, '' if have_arial else '(Arial not installed here; Liberation Sans has the same metrics)')
plt.rcParams.update({'font.family': 'sans-serif', 'font.sans-serif': [FONT, 'Liberation Sans', 'DejaVu Sans'], 'font.size': 10,
                     'axes.labelsize': 11, 'axes.labelweight': 'bold', 'axes.titlesize': 11, 'axes.linewidth': 0.9,
                     'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': False,
                     'xtick.labelsize': 9.5, 'ytick.labelsize': 9.5, 'xtick.major.width': 0.9, 'ytick.major.width': 0.9,
                     'legend.fontsize': 9, 'legend.frameon': False, 'figure.facecolor': 'white', 'axes.facecolor': 'white',
                     'savefig.facecolor': 'white', 'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.fonttype': 'none'})
BLACK, DGREY, MGREY, LGREY, VLGREY, WHITE = '#000000', '#4d4d4d', '#808080', '#b3b3b3', '#dedede', '#ffffff'
# NEJM palette (ggsci 'nejm'): red, blue, orange, green, purple, grey-blue, yellow, pink
NEJM = dict(red='#BC3C29', blue='#0072B5', orange='#E18727', green='#20854E', purple='#7876B1', greyblue='#6F99AD', yellow='#FFDC91', pink='#EE4C97')
C_DER, C_VAL = NEJM['blue'], NEJM['red']          # derivation, validation
INK = '#222222'                                   # lines, whiskers and text

def tint(hex_color, f):
    """Mix a color with white; f = 0 keeps the color, f = 1 gives white."""
    rgb = np.array(matplotlib.colors.to_rgb(hex_color))
    return matplotlib.colors.to_hex(rgb + (1 - rgb) * f)
FIG_W = 7.0

def panel_letters(fig, pairs, size=14, pad=0.012):
    """Place panel letters at the upper left of each axes, left edge aligned with the left edge of its y-axis title."""
    fig.canvas.draw(); r = fig.canvas.get_renderer()
    for ax, letter in pairs:
        lab = ax.yaxis.label
        bb = lab.get_window_extent(r) if lab.get_text() else ax.get_tightbbox(r)
        x = bb.x0 / fig.bbox.width
        y = ax.get_position().y1 + pad
        fig.text(x, y, letter, fontsize=size, fontweight='bold', ha='left', va='bottom')

def savefig(fig, name):
    for ext, kw in [('pdf', {}), ('png', {'dpi': 600}), ('tiff', {'dpi': 600, 'pil_kwargs': {'compression': 'tiff_lzw'}})]:
        fig.savefig(OUT / f'{name}.{ext}', bbox_inches='tight', pad_inches=0.04, **kw)
    plt.close(fig)
    print('saved', name)

LEGENDS = {}

## 1. Cohorts rebuilt from the frozen tables and checked against the locks
Derivation: the locked cohort table `cohort_C1_primary.csv` (v4.1) and the index-admission table. Validation: the v5 rule
(index admissions whose first in-hospital echocardiogram falls in the structured-export window; per-report combined category;
windows from discharge), applied to the frozen report tables of the Kit (v1) and its v1.1 addendum.

In [ ]:
MI = pd.read_parquet(FROZEN / 'KD_master_index_v1_reduced.parquet')
MI['research_patient_id'] = MI['research_patient_id'].astype(str)
yr = MI['index_year']                      # year of the index admission (same definition as Figure S1 of v4.1)
FLOW_D = dict(all_index=len(MI), pre2018=int((yr < 2018).sum()), y2021=int((yr == 2021).sum()), y2025_26=int((yr >= 2025).sum()),
              primary=int(((yr >= 2018) & (yr <= 2024) & (yr != 2021)).sum()))
C1 = pd.read_csv(FROZEN / 'cohort_C1_primary.csv', dtype={'科研患者编号': str, 'index_enc': str})
C1['research_patient_id'] = C1['科研患者编号']
assert set(C1['research_patient_id']) == set(MI.loc[MI['in_C1_primary'] == 1, 'research_patient_id'])
FLOW_D['c1'] = len(C1); FLOW_D['no_inhosp_echo'] = FLOW_D['primary'] - FLOW_D['c1']
U_D = C1[C1['in_max'] == 0].copy()
FLOW_D['inhosp_involvement'] = int((C1['in_max'] >= 1).sum()); FLOW_D['uncomplicated'] = len(U_D)
Q_D = U_D[U_D['post30_n'] > 0]
FLOW_D['q1'] = len(Q_D); FLOW_D['no_q1'] = len(U_D) - len(Q_D)
W2_D = Q_D[(Q_D['post30_cat_max'] == 0) & Q_D['d31_90_cat_max'].notna()]
W3_D = W2_D[(W2_D['d31_90_cat_max'] == 0) & W2_D['d91_365_cat_max'].notna()]
W4_D = W3_D[(W3_D['d91_365_cat_max'] == 0) & W3_D['gt365_cat_max'].notna()]
FLOW_D.update(w2=len(W2_D), w3=len(W3_D), w4=len(W4_D),
              q1_to_w2_finding=int((Q_D['post30_cat_max'] >= 1).sum()),
              q1_to_w2_noecho=int(((Q_D['post30_cat_max'] == 0) & Q_D['d31_90_cat_max'].isna()).sum()),
              w2_to_w3_finding=int((W2_D['d31_90_cat_max'] >= 1).sum()),
              w2_to_w3_noecho=int(((W2_D['d31_90_cat_max'] == 0) & W2_D['d91_365_cat_max'].isna()).sum()),
              w3_to_w4_finding=int((W3_D['d91_365_cat_max'] >= 1).sum()),
              w3_to_w4_noecho=int(((W3_D['d91_365_cat_max'] == 0) & W3_D['gt365_cat_max'].isna()).sum()))
print('derivation flow:', FLOW_D)
# checks against the locks
p = KN4['primary']
assert FLOW_D['c1'] == p['C1_n'] and FLOW_D['uncomplicated'] == p['uncomplicated_n'] and FLOW_D['q1'] == p['surv_n'] and FLOW_D['no_q1'] == p['no_echo_30d']
assert FLOW_D['inhosp_involvement'] == sum(v for k, v in p['inhosp_cat'].items() if k != 'None')
assert FLOW_D['y2021'] == KN41['MOD']['M1a_overview']['index_2021'] and FLOW_D['primary'] == KN41['MOD']['M4b']['summary']['index_admissions']
for k, step in [('w2', '31–90 d after uncomplicated'), ('w3', '91–365 d after normal'), ('w4', '>365 d after normal')]:
    n_lock = int(T2[(T2['scope'] == 'primary') & T2['step'].str.startswith(step)]['n'].iloc[0])
    assert FLOW_D[k] == n_lock, (k, FLOW_D[k], n_lock)
assert_lock(nfmt(FLOW_D['c1']), nfmt(FLOW_D['uncomplicated']), nfmt(FLOW_D['q1']), nfmt(FLOW_D['no_q1']), nfmt(FLOW_D['primary']), nfmt(FLOW_D['y2021']))
print('derivation flow matches the locks')

# validation cohort (v5 rule)
E = pd.read_parquet(KIT / 'KD_echo_reports_AHA_v1.parquet').drop_duplicates('report_key', keep='first')
E['research_patient_id'] = E['research_patient_id'].astype(str); E['exam_date'] = pd.to_datetime(E['exam_time']).dt.normalize()
MEAS = pd.read_parquet(FROZEN / 'KD_echo_measure_2025_2026_dates.parquet')
EXPORT_START, DATA_END = MEAS['exam_date'].min(), MEAS['exam_date'].max()
COMB = pd.read_parquet(ADD / 'KD_echo_reports_combined_v5.parquet')
MI['dis'] = pd.to_datetime(MI['index_discharge_date']).dt.normalize()
MI['first_inh'] = MI['research_patient_id'].map(E[E['phase'] == 'in_index'].groupby('research_patient_id')['exam_date'].min())
CAND = MI[MI['first_inh'] >= EXPORT_START].set_index('research_patient_id')

def day_max(df, cat='combined_cat'):
    d = df[df[cat].notna()]
    return d.groupby(['research_patient_id', 'exam_date'])[cat].max().reset_index()

R = COMB[COMB['research_patient_id'].isin(CAND.index)]
DM = {ph: day_max(R[R['phase'] == ph]) for ph in ['in_index', 'post30', 'd31_90', 'd91_365', 'gt365']}
CH = pd.DataFrame(index=CAND.index); CH['dis'] = CAND['dis']
CH['inmax'] = DM['in_index'].groupby('research_patient_id')['combined_cat'].max()
CH['inhosp_reports'] = R[(R['phase'] == 'in_index') & R['combined_cat'].notna()].groupby('research_patient_id').size()
p30 = DM['post30'].sort_values(['research_patient_id', 'exam_date'])
CH['q1_cat'] = p30.groupby('research_patient_id')['combined_cat'].first()
CH['p30_max'] = p30.groupby('research_patient_id')['combined_cat'].max()
for ph, nm in [('d31_90', 'w2_max'), ('d91_365', 'w3_max'), ('gt365', 'w4_max')]:
    CH[nm] = DM[ph].groupby('research_patient_id')['combined_cat'].max()
END30, END90, END365 = DATA_END - pd.Timedelta(days=30), DATA_END - pd.Timedelta(days=90), DATA_END - pd.Timedelta(days=365)
V_V = CH[CH['inmax'].notna()]
U_V = V_V[V_V['inmax'] == 0]
U30_V = U_V[U_V['dis'] <= END30]
Q_V = U30_V[U30_V['q1_cat'].notna()]
W2_V = U_V[(U_V['dis'] <= END90) & (U_V['p30_max'] == 0) & U_V['w2_max'].notna()]
W3_V = U_V[(U_V['dis'] <= END365) & (U_V['p30_max'] == 0) & (U_V['w2_max'] == 0) & U_V['w3_max'].notna()]
FLOW_V = dict(cand=len(CAND), v=len(V_V), no_class=len(CAND) - len(V_V), inhosp_involvement=int((V_V['inmax'] >= 1).sum()), uncomplicated=len(U_V),
              short_fu=len(U_V) - len(U30_V), u30=len(U30_V), q1=len(Q_V), no_q1=len(U30_V) - len(Q_V), w2=len(W2_V), w3=len(W3_V),
              q1_to_w2_finding=int((Q_V['p30_max'] >= 1).sum()),
              q1_to_w2_short=int(((Q_V['p30_max'] == 0) & (Q_V['dis'] > END90)).sum()),
              q1_to_w2_noecho=int(((Q_V['p30_max'] == 0) & (Q_V['dis'] <= END90) & Q_V['w2_max'].isna()).sum()),
              w2_to_w3_finding=int((W2_V['w2_max'] >= 1).sum()),
              w2_to_w3_short=int(((W2_V['w2_max'] == 0) & (W2_V['dis'] > END365)).sum()),
              w2_to_w3_noecho=int(((W2_V['w2_max'] == 0) & (W2_V['dis'] <= END365) & W2_V['w3_max'].isna()).sum()),
              start=EXPORT_START.date(), end=DATA_END.date())
print('validation flow:', FLOW_V)
c = KN5['cohort']
assert (FLOW_V['cand'], FLOW_V['v'], FLOW_V['uncomplicated'], FLOW_V['u30'], FLOW_V['q1'], FLOW_V['w2'], FLOW_V['w3']) == \
       (c['candidates'], c['V'], c['uncomplicated'], c['complete_30d'], c['Q1'], c['W2'], c['W3'])
assert FLOW_V['inhosp_involvement'] == sum(v for k, v in c['inhospital_max'].items() if k != 'none')
assert int((Q_V['q1_cat'] >= 2).sum()) == KN5['comparisons']['P1']['v5_events']
assert W2_V.index.isin(Q_V.index).all() and W3_V.index.isin(W2_V.index).all(), 'validation windows are not nested'
assert FLOW_V['q1'] - FLOW_V['w2'] == FLOW_V['q1_to_w2_finding'] + FLOW_V['q1_to_w2_short'] + FLOW_V['q1_to_w2_noecho']
assert FLOW_V['w2'] - FLOW_V['w3'] == FLOW_V['w2_to_w3_finding'] + FLOW_V['w2_to_w3_short'] + FLOW_V['w2_to_w3_noecho']
for a, b, parts in [('q1', 'w2', ['q1_to_w2_finding', 'q1_to_w2_noecho']), ('w2', 'w3', ['w2_to_w3_finding', 'w2_to_w3_noecho']),
                    ('w3', 'w4', ['w3_to_w4_finding', 'w3_to_w4_noecho'])]:
    assert FLOW_D[a] - FLOW_D[b] == sum(FLOW_D[p] for p in parts), (a, b)
assert_lock(nfmt(FLOW_V['cand']), nfmt(FLOW_V['v']), nfmt(FLOW_V['uncomplicated']), nfmt(FLOW_V['u30']), nfmt(FLOW_V['q1']))
print('validation flow matches the locks')

## 2. Helpers for the composite figures
Kaplan–Meier estimation is that of section 3 (copied from the locked v4.1 notebook); the log-rank test is the standard k-group test.
Before a figure is saved, `check_layout` stops the run if two pieces of text overlap, if a legend lies on a line, marker, whisker,
bar, confidence band or text, if a label inside a panel lies on a line or whisker, if anything falls outside the canvas, or if the
saved figure would be wider than 7.0 in or taller than 9.0 in.

In [ ]:
from scipy.stats import chi2 as _chi2
from matplotlib import transforms as mtransforms
from matplotlib.text import Text
from matplotlib.lines import Line2D
from matplotlib.collections import LineCollection, PolyCollection
from matplotlib.patches import Rectangle as _Rect
LEG_S, FIG_S = {}, {}                              # supplemental figure legends and file stems
EXTRA = {}                                         # descriptive numbers quoted in the manuscript (written to a JSON at the end)
MAX_W, MAX_H, PAD_IN = 7.0, 9.0, 0.04              # largest saved figure (in) and the padding savefig adds around the content

def parse_kn(s):
    """'67/221 (30.3%; 24.6–36.7)' -> (67, 221)."""
    m = re.match(r'\s*(\d+)/(\d+)', str(s))
    return int(m.group(1)), int(m.group(2))

def logrank(time, event, group):
    """k-group log-rank test (chi-square with k - 1 df)."""
    t = np.asarray(time, float); e = np.asarray(event, int); g = np.asarray(group)
    levels = list(pd.unique(g)); k = len(levels)
    O = np.zeros(k); Ex = np.zeros(k); V = np.zeros((k, k))
    for u in np.unique(t[e == 1]):
        at = t >= u; n = at.sum(); d = ((t == u) & (e == 1)).sum()
        nj = np.array([(at & (g == l)).sum() for l in levels]); dj = np.array([((t == u) & (e == 1) & (g == l)).sum() for l in levels])
        O += dj; Ex += d * nj / n
        if n > 1:
            V += d * (n - d) / (n - 1) * (np.diag(nj / n) - np.outer(nj / n, nj / n))
    oe = (O - Ex)[:-1]
    stat = float(oe @ np.linalg.solve(V[:-1, :-1], oe))
    return stat, k - 1, float(_chi2.sf(stat, k - 1))

def p_txt(p):
    return 'P < 0.001' if p < 0.001 else (f'P = {p:.3f}' if p < 0.1 else f'P = {p:.2f}')

def ci_lock(kmdf, q):
    """As ci_txt, but rounded to 2 decimals first and then to 1, as the locked Table 6 and Results text were (v4)."""
    s, l, h = km_at(kmdf, q)
    v = [round(x * 100, 2) for x in (1 - s, 1 - h, 1 - l)]
    return f'{v[0]:.1f}% ({v[1]:.1f}–{v[2]:.1f})'

FU_REG = 1095.0
def tte_below(reports, cat, dis, thr):
    """Days from discharge to the first post-discharge report with category <= thr (censored at the last report and at 1,095 days)."""
    r = reports.copy()
    r['d'] = (r['exam_date'] - dis.reindex(r['research_patient_id']).values).dt.days
    ev = r[r[cat] <= thr].groupby('research_patient_id')['d'].min(); last = r.groupby('research_patient_id')['d'].max()
    t = ev.reindex(last.index).fillna(last)
    out = pd.DataFrame({'t': t.values, 'e': last.index.isin(ev.index).astype(int)}, index=last.index)
    out.loc[out['t'] > FU_REG, 'e'] = 0; out['t'] = out['t'].clip(upper=FU_REG)
    return out

def km_panel(ax, groups, tmax, ticks, ylab, ylim, yticks, risk_rows_in=0.17, risk_gap_in=0.42, fs=8, bands=True, legend=True,
             leg_kw=None, risk_x_in=0.24, band_alpha=0.12):
    """Cumulative incidence (1 − KM) for several groups, legend above the axes (or as leg_kw says), number-at-risk table under it."""
    for lab, d, col in groups:
        k = km(d['t'], d['e']); t = k['t'].values
        ax.step(t, (1 - k['S'].values) * 100, where='post', color=col, lw=1.7, label=lab, zorder=3)
        if bands:
            lo, hi = zip(*[(1 - km_at(k, tt)[2], 1 - km_at(k, tt)[1]) for tt in t])
            ax.fill_between(t, np.array(lo) * 100, np.array(hi) * 100, step='post', color=col, alpha=band_alpha, linewidth=0, zorder=1)
    ax.set_xlim(0, tmax); ax.set_xticks(ticks); ax.set_xticklabels([f'{q:,}' for q in ticks])
    ax.set_ylim(*ylim); ax.set_yticks(yticks)
    ax.set_xlabel('Days after discharge'); ax.set_ylabel(ylab)
    if legend:
        kw = dict(loc='lower right', bbox_to_anchor=(1.0, 1.0), ncol=len(groups), handlelength=1.2, handletextpad=0.45,
                  borderaxespad=0.15, columnspacing=0.9, fontsize=fs)
        kw.update(leg_kw or {})
        ax.legend(**kw)
    h_in = ax.get_position().height * ax.figure.get_figheight()
    row0 = -(risk_gap_in / h_in); drow = risk_rows_in / h_in
    risk_x = -risk_x_in / (ax.get_position().width * ax.figure.get_figwidth())   # row labels end left of the numbers at time 0
    ax.text(risk_x, row0, 'No. at risk', transform=ax.transAxes, ha='right', va='center', fontsize=fs, fontweight='bold', color=INK)
    for i, (lab, d, col) in enumerate(groups):
        yy = row0 - (i + 1) * drow
        ax.text(risk_x, yy, lab, transform=ax.transAxes, ha='right', va='center', fontsize=fs, color=col, fontweight='bold')
        for q in ticks:
            nr = int((np.asarray(d['t']) >= q).sum())
            ax.text(q / tmax, yy, f'{nr:,}' if nr else '–', transform=ax.transAxes, ha='center', va='center', fontsize=fs, color=INK)

# ---- layout checks
def _overlap(a, b, tol=0.5):
    return min(a.x1, b.x1) - max(a.x0, b.x0) > tol and min(a.y1, b.y1) - max(a.y0, b.y0) > tol

def _texts_of_axes(ax):
    if not ax.get_visible():
        return []
    out = list(ax.texts) + [ax.title, ax._left_title, ax._right_title]
    for axis in (ax.xaxis, ax.yaxis):
        if ax.axison and axis.get_visible():
            out.append(axis.label)
            for tk in axis._update_ticks():               # only the ticks that are drawn
                out += [tk.label1, tk.label2]
    leg = ax.get_legend()
    if leg is not None:
        out += list(leg.get_texts()) + [leg.get_title()]
    for ch in ax.child_axes:
        out += _texts_of_axes(ch)
    return out

def _drawn_texts(fig):
    out = list(fig.texts)
    for leg in fig.legends:
        out += list(leg.get_texts()) + [leg.get_title()]
    for ax in fig.axes:
        out += _texts_of_axes(ax)
    seen, res = set(), []
    for t in out:
        if id(t) not in seen and t.get_visible() and t.get_text().strip():
            seen.add(id(t)); res.append(t)
    return res

def _line_hits(ln, bb):
    """True if the drawn line (segments, or markers with their size) of a Line2D passes through the display box bb."""
    if not ln.get_visible() or len(ln.get_xydata()) == 0:
        return False
    tr = ln.get_transform()
    if ln.get_linestyle() not in ('None', 'none', '', ' ') and len(ln.get_xydata()) > 1:
        if tr.transform_path(ln.get_path()).intersects_bbox(bb, filled=False):
            return True
    if ln.get_marker() not in (None, 'None', 'none', '', ' '):
        xy = tr.transform(np.asarray(ln.get_xydata(), float)); m = ln.get_markersize() * ln.figure.dpi / 72 / 2
        if np.any((xy[:, 0] > bb.x0 - m) & (xy[:, 0] < bb.x1 + m) & (xy[:, 1] > bb.y0 - m) & (xy[:, 1] < bb.y1 + m)):
            return True
    return False

def _segments_hit(col, bb):
    tr = col.get_transform()
    return any(tr.transform_path(p).intersects_bbox(bb, filled=False) for p in col.get_paths())

def _band_hits(col, bb):
    tr = col.get_transform()
    return any(tr.transform_path(p).intersects_bbox(bb, filled=True) for p in col.get_paths())

def check_layout(fig, name):
    fig.canvas.draw(); r = fig.canvas.get_renderer()
    problems = []
    texts = [(t, t.get_window_extent(r)) for t in _drawn_texts(fig)]
    W, H = fig.bbox.width, fig.bbox.height
    for t, b in texts:                                     # 1. everything inside the canvas
        if b.x0 < -1 or b.y0 < -1 or b.x1 > W + 1 or b.y1 > H + 1:
            problems.append(f'text outside the canvas: "{t.get_text()[:40]}"')
    for i in range(len(texts)):                            # 2. no two texts overlap or touch (clearance 1.5 px = 0.015 in)
        for j in range(i + 1, len(texts)):
            if _overlap(texts[i][1], texts[j][1], tol=-1.5):
                problems.append(f'text overlap: "{texts[i][0].get_text()[:40]}" | "{texts[j][0].get_text()[:40]}"')
    legends = [(ax.get_legend(), [ax]) for ax in fig.axes if ax.get_legend() is not None] + [(lg, fig.axes) for lg in fig.legends]
    def _within(art, lb, ax):
        """The part of the legend box in which the artist can be visible (artists clipped to the axes are not drawn outside it)."""
        return mtransforms.Bbox.intersection(lb, ax.bbox) if art.get_clip_on() else lb
    for leg, axes in legends:                              # 3. a legend covers no data and no text
        lb = leg.get_window_extent(r); own = set(map(id, list(leg.get_texts()) + [leg.get_title()]))
        for ax in axes:
            if any((b := _within(ln, lb, ax)) is not None and _line_hits(ln, b) for ln in ax.lines):
                problems.append(f'legend covers a line or marker ({ax.get_ylabel() or "axes"})')
            for col in ax.collections:
                b = _within(col, lb, ax)
                if b is None:
                    continue
                if isinstance(col, LineCollection) and _segments_hit(col, b):
                    problems.append(f'legend covers a whisker ({ax.get_ylabel() or "axes"})')
                elif isinstance(col, PolyCollection) and _band_hits(col, b):
                    problems.append(f'legend covers a confidence band ({ax.get_ylabel() or "axes"})')
            if any(isinstance(p, _Rect) and p.get_visible() and _overlap(p.get_window_extent(r), lb) for p in ax.patches):
                problems.append(f'legend covers a bar ({ax.get_ylabel() or "axes"})')
        for t, b in texts:
            if id(t) not in own and _overlap(b, lb):
                problems.append(f'legend covers the text "{t.get_text()[:30]}"')
    for ax in fig.axes:                                    # 4. a label inside a panel lies on no line and no whisker
        for t in ax.texts:
            if not (t.get_visible() and t.get_text().strip()):
                continue
            tb = t.get_window_extent(r)
            if any(_line_hits(ln, tb) for ln in ax.lines) or any(isinstance(c, LineCollection) and _segments_hit(c, tb) for c in ax.collections):
                problems.append(f'the label "{t.get_text()[:30]}" lies on a line or whisker')
    bb = fig.get_tightbbox(r)                              # 5. saved size
    w, h = bb.width + 2 * PAD_IN, bb.height + 2 * PAD_IN
    if w > MAX_W + 1e-6 or h > MAX_H + 1e-6:
        problems.append(f'saved size {w:.2f} x {h:.2f} in exceeds {MAX_W} x {MAX_H} in')
    problems = sorted(set(problems))
    if problems and os.environ.get('KD_LAYOUT_DEBUG') == '1':        # development only: report and save anyway
        print(f'LAYOUT PROBLEMS in {name}:', *problems, sep='\n  ')
        return w, h
    assert not problems, f'{name}: ' + '; '.join(problems[:10])
    print(f'{name}: layout check passed ({len(texts)} texts; saved size {w:.2f} x {h:.2f} in)')
    return w, h

FIG_SIZE = {}
def save_checked(fig, name):
    FIG_SIZE[name] = check_layout(fig, name)
    savefig(fig, name)

def letter_at(fig, ax, letter, x_from='ylabel', pad_in=0.06, x_in=None):
    """Panel letter at the upper left of ax, aligned with its y-axis title (or with its y tick labels when it has no title)."""
    fig.canvas.draw(); r = fig.canvas.get_renderer()
    if x_in is not None:
        x0 = x_in * fig.dpi
    elif x_from == 'ylabel' and ax.yaxis.label.get_text():
        x0 = ax.yaxis.label.get_window_extent(r).x0
    else:
        x0 = min([t.get_window_extent(r).x0 for t in ax.get_yticklabels() if t.get_text()] + [ax.get_window_extent(r).x0])
    fig.text(x0 / fig.bbox.width, ax.get_position().y1 + pad_in / fig.get_figheight(), letter, fontsize=14, fontweight='bold',
             ha='left', va='bottom')

## 3. Figure 2 — findings at successive post-discharge echocardiograms (derivation beside validation)

In [ ]:
def parse_t2(scope='primary'):
    """Rows of the locked Table2_sequential_yield.csv as (window, outcome) -> (events, n, pct, ci_low, ci_high)."""
    out, window = {}, None
    for _, r in T2[T2['scope'] == scope].iterrows():
        s = r['step'].strip()
        if not s.startswith('…'):
            if s.startswith('First echo ≤30 d after an uncomplicated'):
                window = '1–30'
            elif s.startswith('31–90 d after uncomplicated in-hospital + normal'):
                window = '31–90'
            elif s.startswith('91–365 d after normal'):
                window = '91–365'
            elif s.startswith('>365 d after normal'):
                window = '>365'
            else:
                window = None                      # 31–90 d with or without a first echo: not part of the ladder
            outcome = s.split(':', 1)[1].strip() if ':' in s else ''
        else:
            outcome = s.lstrip('… ').strip()
        if window is None:
            continue
        key = 'aneurysm' if outcome.startswith('aneurysm (Z') or outcome == 'aneurysm' else ('involvement' if outcome.startswith('any involvement') else None)
        if key:
            out[(window, key)] = (int(r['events']), int(r['n']), float(r['pct']), float(r['ci_low']), float(r['ci_high']))
    return out

WIN = ['1–30', '31–90', '91–365', '>365']
DER = parse_t2('primary')
assert len(DER) == 8, DER
VAL = {('1–30', 'aneurysm'): 'P1', ('1–30', 'involvement'): 'S1', ('31–90', 'aneurysm'): 'S4_aneurysm', ('31–90', 'involvement'): 'S4_involvement'}
VAL = {k: (int(V51.loc[v, 'v5_events']), int(V51.loc[v, 'v5_n']), float(V51.loc[v, 'v5_pct']), float(V51.loc[v, 'v5_ci_low']), float(V51.loc[v, 'v5_ci_high']))
       for k, v in VAL.items()}
for o in ['aneurysm', 'involvement']:
    s5 = KN5['S5'][o]
    VAL[('91–365', o)] = (s5['events'], s5['n'], s5['pct'], s5['ci_low'], s5['ci_high'])
# the derivation counts must agree with the cohort rebuilt above and with the lock text
assert DER[('1–30', 'aneurysm')][:2] == (int((Q_D['post30_cat_first'] >= 2).sum()), len(Q_D))
assert DER[('31–90', 'involvement')][:2] == (int((W2_D['d31_90_cat_max'] >= 1).sum()), len(W2_D))
assert VAL[('1–30', 'aneurysm')][:2] == (int((Q_V['q1_cat'] >= 2).sum()), len(Q_V))
assert VAL[('91–365', 'aneurysm')][:2] == (int((W3_V['w3_max'] >= 2).sum()), len(W3_V))
for d in (DER, VAL):
    for (w, o), (k, n, pct, lo, hi) in d.items():
        assert_lock(f'{k}/{nfmt(n)}')
        wl = wilson(k, n)
        assert abs(wl[0] - lo) < 0.006 and abs(wl[1] - hi) < 0.006, ((w, o), wl, lo, hi)

Panel C: cumulative incidence of a newly detected aneurysm after discharge (1 − Kaplan–Meier). Derivation: the locked TTE-1 analysis
of v4.1 (children with an uncomplicated course and any post-discharge echocardiogram up to February 28, 2025; time 0 = discharge;
event = first post-discharge report with Z ≥ 2.5; censored at the last report or at 730 days), reproduced here and checked against
R5. Validation: the same definition applied to the 2025–2026 reports; added after the validation plan was locked (descriptive).

In [ ]:
def km(time, event):
    """Kaplan–Meier estimate with Greenwood variance (copied from KD_echo_surveillance_v4_1)."""
    t = np.asarray(time, float); d = np.asarray(event, int)
    order = np.argsort(t); t = t[order]; d = d[order]
    rows = [dict(t=0.0, n_risk=len(t), n_event=0, S=1.0, se=0.0)]; S = 1.0; gw = 0.0
    for u in np.unique(t[d == 1]):
        n = int((t >= u).sum()); e = int(((t == u) & (d == 1)).sum())
        if n == 0:
            continue
        S *= (1 - e / n); gw += e / (n * (n - e)) if n > e else 0.0
        rows.append(dict(t=u, n_risk=n, n_event=e, S=S, se=S * np.sqrt(gw)))
    if len(t) and t.max() > rows[-1]['t']:
        rows.append(dict(t=float(t.max()), n_risk=int((t >= t.max()).sum()), n_event=0, S=S, se=rows[-1]['se']))
    return pd.DataFrame(rows)

def km_at(kmdf, tq):
    """S and 95% CI (log-log) at time tq (copied from KD_echo_surveillance_v4_1)."""
    r = kmdf[kmdf['t'] <= tq].iloc[-1]; S, se = r['S'], r['se']
    if S <= 0 or S >= 1 or se == 0:
        return S, S, S
    z = 1.96; lg = np.log(-np.log(S)); v = se / (S * abs(np.log(S)))
    return S, np.exp(-np.exp(lg + z * v)), np.exp(-np.exp(lg - z * v))

POST = ['post30', 'd31_90', 'd91_365', 'gt365']
ERA_END, FU_NEW = pd.Timestamp('2025-02-28'), 730.0
assert_lock('Numeric era ends 2025-02-28')

def tte(reports, cat, dis):
    r = reports.copy()
    r['d'] = (r['exam_date'] - dis.reindex(r['research_patient_id']).values).dt.days
    ev = r[r[cat] >= 2].groupby('research_patient_id')['d'].min(); last = r.groupby('research_patient_id')['d'].max()
    t = ev.reindex(last.index).fillna(last)
    return pd.DataFrame({'t': t.values, 'e': last.index.isin(ev.index).astype(int)}, index=last.index)

TD = tte(E[E['aha_best'].notna() & (E['exam_date'] <= ERA_END) & E['phase'].isin(POST) & E['research_patient_id'].isin(U_D['research_patient_id'])],
         'aha_best', MI.set_index('research_patient_id')['dis'])
TD.loc[TD['t'] > FU_NEW, 'e'] = 0; TD['t'] = TD['t'].clip(upper=FU_NEW)
TV = tte(COMB[COMB['combined_cat'].notna() & COMB['phase'].isin(POST) & COMB['research_patient_id'].isin(U_V.index)], 'combined_cat', CH['dis'])
KMD, KMV = km(TD['t'], TD['e']), km(TV['t'], TV['e'])
def ci_txt(kmdf, q):
    s, l, h = km_at(kmdf, q)
    return f'{(1 - s) * 100:.1f}% ({(1 - h) * 100:.1f}–{(1 - l) * 100:.1f})'
for q in [30, 90, 365, 730]:
    assert_lock(f'{q} d {ci_txt(KMD, q)}')
assert_lock(f"(n = {len(TD):,}, {int(TD['e'].sum())} events)")
KMV_SUM = dict(n=len(TV), events=int(TV['e'].sum()), last_event_day=int(TV.loc[TV['e'] == 1, 't'].max()),
               ci30=ci_txt(KMV, 30), ci365=ci_txt(KMV, 365), risk365=int((TV['t'] >= 365).sum()), med_fu=float(TV['t'].median()))
assert KMV_SUM['events'] == KN5['comparisons']['P1']['v5_events']
print('derivation KM reproduces R5;', 'validation:', KMV_SUM)

fig = plt.figure(figsize=(FIG_W, 6.7))
axA = fig.add_axes([0.15, 0.60, 0.33, 0.36]); axB = fig.add_axes([0.635, 0.60, 0.33, 0.36])
axC = fig.add_axes([0.15, 0.17, 0.815, 0.30])
RX = -0.24 / (0.815 * FIG_W)                     # number-at-risk labels end 0.24 in left of the axis
xw = 0.36
for ax, outcome, ylab in [(axA, 'aneurysm', 'Aneurysm (%)'), (axB, 'involvement', 'Any involvement (%)')]:
    for src, col, dx, lab in [(DER, C_DER, -xw / 2, 'Derivation'), (VAL, C_VAL, xw / 2, 'Validation')]:
        for i, w in enumerate(WIN):
            x = i + dx
            if (w, outcome) not in src:
                ax.text(x, 0.08, 'NA', ha='center', va='bottom', fontsize=8, rotation=90, color=INK)
                continue
            k, n, pct, lo, hi = src[(w, outcome)]
            ax.bar(x, pct, width=xw, color=col, edgecolor='none', label=lab if i == 0 else None, zorder=2)
            ax.errorbar(x, pct, yerr=[[pct - lo], [hi - pct]], fmt='none', ecolor=INK, elinewidth=0.9, capsize=2.5, capthick=0.9, zorder=3)
            ax.text(x, hi + 0.12, f'{k}/{nfmt(n)}', ha='center', va='bottom', fontsize=8, rotation=90, color=INK)
    ax.set_xticks(range(len(WIN))); ax.set_xticklabels(WIN); ax.set_xlim(-0.6, len(WIN) - 0.4)
    ax.set_xlabel('Days after discharge'); ax.set_ylabel(ylab); ax.set_ylim(0, 5.0); ax.set_yticks(range(0, 6))
axA.legend(loc='upper right', handlelength=1.2, borderaxespad=0.2)
for kmdf, col, lab in [(KMD, C_DER, 'Derivation'), (KMV, C_VAL, 'Validation')]:
    t = kmdf['t'].values
    axC.step(t, (1 - kmdf['S'].values) * 100, where='post', color=col, lw=1.8, label=lab, zorder=3)
    lo, hi = zip(*[(1 - km_at(kmdf, tt)[2], 1 - km_at(kmdf, tt)[1]) for tt in t])
    axC.fill_between(t, np.array(lo) * 100, np.array(hi) * 100, step='post', color=col, alpha=0.15, linewidth=0, zorder=1)
TQ = [0, 90, 180, 365, 545, 730]
axC.set_xlim(0, FU_NEW); axC.set_ylim(0, 4); axC.set_xticks(TQ); axC.set_yticks(range(0, 5))
axC.set_xlabel('Days after discharge'); axC.set_ylabel('New aneurysm (%)')
axC.legend(loc='lower right', handlelength=1.6)
axC.text(RX, -0.25, 'No. at risk', transform=axC.transAxes, ha='right', va='center', fontsize=8.5, fontweight='bold', color=INK)
for i, (tt, lab, col) in enumerate([(TD['t'], 'Derivation', C_DER), (TV['t'], 'Validation', C_VAL)]):
    yy = -0.37 - i * 0.12
    axC.text(RX, yy, lab, transform=axC.transAxes, ha='right', va='center', fontsize=8.5, color=col, fontweight='bold')
    for q in TQ:
        nr = int((np.asarray(tt) >= q).sum())
        axC.text(q / FU_NEW, yy, f'{nr:,}' if nr else '–', transform=axC.transAxes, ha='center', va='center', fontsize=8.5, color=INK)
panel_letters(fig, [(axA, 'A'), (axB, 'B'), (axC, 'C')])
save_checked(fig, 'Figure2_successive_postdischarge_echocardiograms')

pP = {k: float(V51.loc[k, 'predictive_P']) for k in ['P1', 'S1', 'S2', 'S4_aneurysm', 'S4_involvement']}
def fp(p):
    return 'P < 0.001' if p < 0.001 else f'P = {p:.3f}' if p < 0.05 else f'P = {p:.2f}'
s2 = V51.loc['S2']
LEGENDS['Figure 2'] = (
    "Figure 2. Findings at successive echocardiograms after discharge in children with an uncomplicated in-hospital course. "
    "Blue, derivation cohort (2018–2024); red, temporal validation cohort (2025–2026). A, coronary artery aneurysm (maximal Z ≥ 2.5) "
    "and B, any coronary involvement (maximal Z ≥ 2) by window after discharge. The 1–30-day window counts the first echocardiogram "
    "after discharge; each later window includes children in whom every echocardiogram in the earlier windows was normal and who had "
    "an echocardiogram in that window, and counts the largest finding in it. Bars show percentages, whiskers Wilson 95% CIs, and "
    "labels children with the finding/children examined; NA, not assessed (validation follow-up too short). Validation against "
    f"derivation, 2-sided beta-binomial predictive test: aneurysm 1–30 days, {fp(pP['P1'])}; any involvement 1–30 days, {fp(pP['S1'])}; "
    f"aneurysm 31–90 days, {fp(pP['S4_aneurysm'])}; and any involvement 31–90 days, {fp(pP['S4_involvement'])} (this comparison was not "
    f"prespecified). Medium or larger aneurysm (Z ≥ 5) at the first echocardiogram: derivation {int(s2['v4_events'])}/{nfmt(s2['v4_n'])} "
    f"({s2['v4_pct']:.2f}%), validation {int(s2['v5_events'])}/{nfmt(s2['v5_n'])} ({fp(pP['S2'])}). C, cumulative incidence of a newly "
    "detected aneurysm after discharge (1 − Kaplan–Meier) in children with an uncomplicated course and at least 1 echocardiogram after "
    f"discharge (derivation n = {len(TD):,}, {int(TD['e'].sum())} aneurysms within 730 days; validation n = {KMV_SUM['n']:,}, "
    f"{KMV_SUM['events']} aneurysms). Time 0 is discharge; an aneurysm is dated at the first echocardiogram that showed it, so the early "
    "rise partly reflects the timing of the first follow-up visit; follow-up is censored at the last echocardiogram (derivation also at "
    f"730 days). Shaded bands, 95% CIs. Derivation: {ci_txt(KMD, 30)} at 30 days, {ci_txt(KMD, 365)} at 1 year, and {ci_txt(KMD, 730)} "
    f"at 2 years. Validation (curve added after the validation plan was locked): all {KMV_SUM['events']} aneurysms within "
    f"{KMV_SUM['last_event_day']} days of discharge; {ci_txt(KMV, 365)} at 1 year ({KMV_SUM['risk365']} children still followed).")
print(LEGENDS['Figure 2'])

## 4. Data for Figure 3C (course of first-visit aneurysms) and Figure 4C (window of first documentation)
Figure 4C uses the locked Table 3a (derivation cohort C1). Figure 3C is a descriptive cut of the frozen per-report table: category at
detection and category of the last report more than 30 days after discharge (echocardiograms up to the end of the numeric era,
February 28, 2025, as in the locked analysis); its totals are asserted against the locked fate counts. Validation: locked
TableV5_4 (events one by one).

In [ ]:
t3 = T3A[T3A['scope'] == 'primary'].set_index('first_documented_in')
FIND = [('involvement_Z2', 'Any\ninvolvement'), ('aneurysm_Z2.5', 'Aneurysm'), ('medium_or_larger_Z5', 'Medium\nor larger')]
F3A = {}
for col, lab in FIND:
    tot = int(t3[col].sum()); w1 = int(t3.loc['W1 1–30 d', col])
    later = int(t3.loc[['W2 31–90 d', 'W3 91–365 d', 'W4 >365 d'], col].sum())
    F3A[col] = dict(total=tot, w1=w1, later=later, inhosp=int(t3.loc['index admission (or before)', col]))
assert F3A['aneurysm_Z2.5']['total'] == KN4['primary']['aneurysm_ever'] and F3A['aneurysm_Z2.5']['w1'] + F3A['aneurysm_Z2.5']['later'] == KN4['primary']['aneurysm_first_after_discharge']
assert_lock(f"involvement: {F3A['involvement_Z2']['total']} children", f"aneurysm: {F3A['aneurysm_Z2.5']['total']} children",
            f"medium or larger: {F3A['medium_or_larger_Z5']['total']} children")

# Panel B, derivation: the 26 aneurysms at the first 1-30-d echocardiogram after an uncomplicated course
ERA_END = pd.Timestamp('2025-02-28')
assert_lock('Numeric era ends 2025-02-28')
na_ids = set(Q_D.loc[Q_D['post30_cat_first'] >= 2, 'research_patient_id'])
EP = E[E['aha_best'].notna() & (E['exam_date'] <= ERA_END) & E['research_patient_id'].isin(na_ids) & E['phase'].isin(['d31_90', 'd91_365', 'gt365'])]
EP = EP.sort_values(['research_patient_id', 'exam_time'], kind='mergesort')
later = EP.groupby('research_patient_id')['aha_best'].agg(['max', 'last'])
fate = KN4['fate']
assert len(na_ids) == fate['n'] and len(later) == fate['with_later_echo']
assert int((later['last'] <= 1).sum()) == fate['last_below_2_5'] and int((later['max'] >= 3).sum()) == fate['later_max_medium_plus']
det_D = Q_D.loc[Q_D['post30_cat_first'] >= 2, 'post30_cat_first'].astype(int).value_counts().to_dict()
assert det_D == {{'Small': 2, 'Medium': 3, 'Large/giant': 4}[k]: v for k, v in fate['category_at_detection'].items()}
last_D = later['last'].astype(int).value_counts().to_dict(); last_D[-1] = len(na_ids) - len(later)
# time from discharge to the last report after day 30 (days), for the legend
MI_dis = MI.set_index('research_patient_id')['dis']
last_day_D = (EP.groupby('research_patient_id')['exam_date'].max() - MI_dis.reindex(later.index)).dt.days
# validation: TableV5_4
assert len(V54) == KN5['comparisons']['P1']['v5_events']
CATN = {'none': 0, 'dilation only': 1, 'small': 2, 'medium': 3, 'large/giant': 4}
det_V = V54['category'].str.lower().map(CATN).value_counts().to_dict()
lv = np.where(V54['last_echo_day_after_discharge'] > 30, V54['last_echo_category'].str.lower().map(CATN), -1)
last_V = pd.Series(lv).value_counts().to_dict()
print('derivation at detection', det_D, '| last report >30 d', last_D, '| median day of that report', float(last_day_D.median()))
print('validation at detection', det_V, '| last report >30 d', last_V)

## 5. Data for Figures 5A and 5D — risk factors and risk-based selection for the first post-discharge echocardiogram
Derivation: locked Table 4a (rates by risk-factor count) and Table M3b (strategies, bootstrap CIs); children discharged with Z < 2.5
and a first echocardiogram 1–30 days after discharge. Validation: locked v5 section S7.

In [ ]:
t4a = T4A.set_index('stratum')
CNT_LAB = ['0', '1', '≥2']
RF_D = {lab: (int(t4a.loc[s, 'events']), int(t4a.loc[s, 'n'])) for lab, s in zip(CNT_LAB, ['0 factors', '1 factor', '≥2 factors'])}
RF_V = {lab: tuple(int(x) for x in KN5['S7']['by_count'][k].split('/')) for lab, k in zip(CNT_LAB, ['0', '1', '2'])}
assert sum(n for _, n in RF_D.values()) == int(M3B.loc[0, 'n']) and sum(k for k, _ in RF_D.values()) == int(M3B.loc[0, 'aneurysms'])
assert sum(n for _, n in RF_V.values()) == KN5['S7']['children'] and sum(k for k, _ in RF_V.values()) == KN5['S7']['aneurysms']
for lab, (k, n) in RF_D.items():
    assert_lock(f'{k}/{n}')
for lab, (k, n) in RF_V.items():
    assert_lock(f'{k}/{n}')

def ci_pair(s):
    a, b = str(s).replace('−', '-').split('–')
    return float(a), float(b)

STRAT = []
for _, r in M3B.iterrows():
    if r['strategy'].startswith('S4'):
        continue                                   # exploratory rule; in the supplementary table only
    xl, xh = ci_pair(r['echoes_avoided_pct_ci']); yl, yh = ci_pair(r['sensitivity_pct_ci'])
    STRAT.append(dict(key=r['strategy'][:2], x=float(r['echoes_avoided_pct']), xl=xl, xh=xh, y=float(r['sensitivity_pct']), yl=yl, yh=yh,
                      det=int(r['detected']), tot=int(r['aneurysms']), echoes=int(r['echoes']), missed=r['missed_by_size']))
s7 = KN5['S7']
VAL_PT = dict(x=s7['echoes_avoided_pct'], y=s7['sensitivity']['pct'], yl=s7['sensitivity']['ci_low'], yh=s7['sensitivity']['ci_high'],
              det=s7['detected'], tot=s7['aneurysms'], echoes=s7['echoes_if_ge1'])
assert_lock(f"avoided {STRAT[1]['x']:.1f}%", f"sensitivity {STRAT[1]['y']:.1f}%", f"(avoided {VAL_PT['x']:.1f}%)")

## 6. Table 2 — aneurysm at the first post-discharge echocardiogram, children discharged with Z < 2.5 (locked Table 4b)

In [ ]:
mv = T4B[(T4B['model'] == 'multivariable') & (T4B['population'] == 'Z<2.5 in hospital')].set_index('variable')
uv = T4B[(T4B['model'] == 'univariable') & (T4B['population'] == 'Z<2.5 in hospital')].set_index('variable')
N_T2, EV_T2 = int(mv['n'].iloc[0]), int(mv['events'].iloc[0])
REF = {'In-hospital dilation only (Z 2–<2.5)': ('In-hospital dilation only (Z 2 to <2.5)', 'Z < 2 throughout'),
       'Age <12 months': ('Age <12 mo', '≥12 mo'), 'Male sex': ('Male', 'Female'), 'IVIG resistance': ('IVIG resistance', 'No resistance'),
       'Discharge CRP >20 mg/L': ('Discharge CRP >20 mg/L', '≤20 mg/L'), 'Discharge WBC >12 ×10⁹/L': ('Discharge WBC >12 × 10⁹/L', '≤12 × 10⁹/L'),
       '≥2 in-hospital echoes': ('≥2 in-hospital echocardiograms', '1')}
def pct_ci(k, n):
    lo, hi = wilson(k, n)
    return f'{k}/{n:,} ({k / n * 100:.1f}; {lo:.1f}–{hi:.1f})'
def fmt_p(p):
    return '<0.001' if p < 0.001 else f'{p:.3f}' if p < 0.1 else f'{p:.2f}'
rows = []
for var, (lab, ref) in REF.items():
    r, u = mv.loc[var], uv.loc[var]
    kw, nw = int(r['events_with']), int(r['n_with'])
    kwo, nwo = int(r['events_without']), N_T2 - nw
    assert kw + kwo == EV_T2
    rows.append({'Characteristic': lab, 'Reference': ref,
                 'Aneurysm with the characteristic, n/N (%; 95% CI)': pct_ci(kw, nw),
                 'Aneurysm in the reference group, n/N (%; 95% CI)': pct_ci(kwo, nwo),
                 'Unadjusted OR (95% CI)': f"{u['OR']:.2f} ({u['CI_low']:.2f}–{u['CI_high']:.2f})",
                 'Adjusted OR (95% CI)': f"{r['OR']:.2f} ({r['CI_low']:.2f}–{r['CI_high']:.2f})", 'P (adjusted)': fmt_p(float(r['p']))})
    assert_lock(f"{var} {r['OR']:.2f} ({r['CI_low']:.2f}–{r['CI_high']:.2f})")
TABLE2 = pd.DataFrame(rows)
TABLE2.to_csv(OUT / 'Table2_aneurysm_at_first_postdischarge_echo.csv', index=False, encoding='utf-8-sig')
print(TABLE2.to_string(index=False))
TABLE_NOTES = {'Table 2': (
    f"Table 2. Aneurysm at the first echocardiogram 1–30 days after discharge in children discharged with a maximal in-hospital Z "
    f"score below 2.5 (n = {N_T2:,}; {EV_T2} aneurysms; derivation cohort). Odds ratios from Firth-penalized logistic regression; "
    "the adjusted model contains all seven characteristics. CIs of proportions are Wilson intervals. CRP, C-reactive protein; IVIG, "
    "intravenous immunoglobulin; OR, odds ratio; WBC, white blood cell count.")}

## 7. Table 1 — characteristics of children with an uncomplicated in-hospital course
Covariates come from the frozen table `KD_JAHA_covariates_v1.parquet` (one row per index admission, research IDs only, no text). It
was built from the hospital exports with the definitions of the locked v4/v4.1 analysis (KD_echo_surveillance_v4_1, sections 1–2;
IVIG courses from orders 36 hours apart; IVIG resistance = a second course, a glucocorticoid started more than 24 hours after the
first IVIG, or infliximab; first and last laboratory values of the admission). The match with the locked Table 4b counts is
asserted before the table is built.

In [ ]:
COV = pd.read_parquet(FROZEN / 'KD_JAHA_covariates_v1.parquet')
COV['research_patient_id'] = COV['research_patient_id'].astype(str)
COV = COV.set_index('research_patient_id')
assert len(COV) == len(MI) and set(COV.index) == set(MI['research_patient_id']), 'index admissions differ from the master index'
print('covariates loaded for', len(COV), 'index admissions')

# definitional check: reproduce the locked Table 4b counts in the 1,702 children discharged with Z < 2.5
P17 = C1[(C1['in_max'] <= 1) & (C1['post30_n'] > 0)].set_index('research_patient_id')
cv = COV.loc[P17.index]
chk = {'In-hospital dilation only (Z 2–<2.5)': P17['in_max'] == 1, 'Age <12 months': P17['age_m'] < 12, 'Male sex': cv['male'] == 1,
       'IVIG resistance': cv['ivig_resistant'], 'Discharge CRP >20 mg/L': cv['CRPc_dis'] > 20, 'Discharge WBC >12 ×10⁹/L': cv['WBC_dis'] > 12,
       '≥2 in-hospital echoes': P17['in_index_n'] >= 2}
for var, s in chk.items():
    assert int(s.sum()) == int(mv.loc[var, 'n_with']), (var, int(s.sum()), int(mv.loc[var, 'n_with']))
assert np.allclose(P17['age_m'].values, cv['age_m'].values, equal_nan=True)
print('covariate definitions reproduce the locked Table 4b counts')

In [ ]:
def smd_cont(a, b):
    a, b = pd.Series(a, dtype=float).dropna(), pd.Series(b, dtype=float).dropna()
    s = np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2)
    return abs(a.mean() - b.mean()) / s if s > 0 else 0.0

def smd_bin(a, b):
    a, b = pd.Series(a).dropna().astype(float), pd.Series(b).dropna().astype(float)
    p1, p2 = a.mean(), b.mean(); s = np.sqrt((p1 * (1 - p1) + p2 * (1 - p2)) / 2)
    return abs(p1 - p2) / s if s > 0 else 0.0

def med_iqr(x, dec=1):
    x = pd.Series(x, dtype=float).dropna()
    return f'{x.median():.{dec}f} ({x.quantile(0.25):.{dec}f}–{x.quantile(0.75):.{dec}f})'

def n_pct(x):
    x = pd.Series(x).dropna().astype(bool)
    return f'{int(x.sum()):,} ({x.mean() * 100:.1f})'

def frame(ids, echo_counts):
    f = COV.loc[list(ids)].copy()
    f['inhosp_echoes'] = pd.Series(echo_counts).reindex(f.index).values
    f['age_lt12'] = f['age_m'] < 12
    f['crp_dis_gt20'] = np.where(f['CRPc_dis'].notna(), f['CRPc_dis'] > 20, np.nan)
    f['wbc_dis_gt12'] = np.where(f['WBC_dis'].notna(), f['WBC_dis'] > 12, np.nan)
    f['echo_ge2'] = f['inhosp_echoes'] >= 2
    inflam = ((f['CRPc_dis'] > 20) | (f['WBC_dis'] > 12)).fillna(False)
    f['rf_ge1'] = f['age_lt12'] | f['ivig_resistant'].astype(bool) | inflam
    return f

GROUPS = [('Derivation: echocardiogram 1–30 d', frame(Q_D['research_patient_id'], C1.set_index('research_patient_id')['in_index_n'])),
          ('Derivation: no echocardiogram 1–30 d', frame(U_D.loc[U_D['post30_n'] == 0, 'research_patient_id'], C1.set_index('research_patient_id')['in_index_n'])),
          ('Validation: echocardiogram 1–30 d', frame(Q_V.index, CH['inhosp_reports'])),
          ('Validation: no echocardiogram 1–30 d', frame(U30_V.index[U30_V['q1_cat'].isna()], CH['inhosp_reports']))]
assert [len(g) for _, g in GROUPS] == [FLOW_D['q1'], FLOW_D['no_q1'], FLOW_V['q1'], FLOW_V['no_q1']]

VARS = [('Age, mo', 'age_m', 'c', 1), ('Age <12 mo', 'age_lt12', 'b', None), ('Male', 'male', 'b', None),
        ('Incomplete Kawasaki disease', 'incomplete_kd', 'b', None), ('Fever days at admission', 'fever_days_adm', 'c', 0),
        ('Illness day of the first IVIG', 'fever_to_ivig', 'c', 1), ('IVIG given', 'ivig_given', 'b', None),
        ('IVIG resistance', 'ivig_resistant', 'b', None), ('Systemic glucocorticoid', 'ster_any', 'b', None),
        ('Length of stay, d', 'LOS', 'c', 0), ('In-hospital echocardiograms, n', 'inhosp_echoes', 'c', 0),
        ('≥2 in-hospital echocardiograms', 'echo_ge2', 'b', None),
        ('WBC at admission, × 10⁹/L', 'WBC_adm', 'c', 1), ('Hemoglobin at admission, g/L', 'Hb_adm', 'c', 0),
        ('Platelets at admission, × 10⁹/L', 'PLT_adm', 'c', 0), ('CRP at admission, mg/L', 'CRPc_adm', 'c', 1),
        ('Albumin at admission, g/L', 'ALB_adm', 'c', 1), ('ALT at admission, U/L', 'ALT_adm', 'c', 0),
        ('WBC at discharge, × 10⁹/L', 'WBC_dis', 'c', 1), ('CRP at discharge, mg/L', 'CRPc_dis', 'c', 1),
        ('Platelets at discharge, × 10⁹/L', 'PLT_dis', 'c', 0), ('Discharge CRP >20 mg/L', 'crp_dis_gt20', 'b', None),
        ('Discharge WBC >12 × 10⁹/L', 'wbc_dis_gt12', 'b', None), ('≥1 risk factor', 'rf_ge1', 'b', None),
        ('Readmitted within 30 d', 'readm30', 'b', None)]
rows = []
for lab, col, kind, dec in VARS:
    r = {'Characteristic': lab + (', median (IQR)' if kind == 'c' else ', n (%)')}
    for gname, g in GROUPS:
        r[f'{gname} (n = {len(g):,})'] = med_iqr(g[col], dec) if kind == 'c' else n_pct(g[col])
        r[f'available: {gname}'] = f"{int(g[col].notna().sum()):,}/{len(g):,}"
    f = smd_cont if kind == 'c' else smd_bin
    r['SMD derivation'] = f'{f(GROUPS[0][1][col], GROUPS[1][1][col]):.2f}'
    r['SMD validation'] = f'{f(GROUPS[2][1][col], GROUPS[3][1][col]):.2f}'
    rows.append(r)
TABLE1 = pd.DataFrame(rows)
TABLE1.to_csv(OUT / 'Table1_characteristics_uncomplicated_course.csv', index=False, encoding='utf-8-sig')
show = TABLE1[[c for c in TABLE1.columns if not c.startswith('available')]]
print(show.to_string(index=False))
low = {lab: [r[c] for c in TABLE1.columns if c.startswith('available') and int(r[c].split('/')[0].replace(',', '')) < 0.95 * int(r[c].split('/')[1].replace(',', ''))]
       for lab, r in zip(TABLE1['Characteristic'], TABLE1.to_dict('records'))}
low = {k: v for k, v in low.items() if v}
print('variables with <95% availability in a group:', low)
TABLE_NOTES['Table 1'] = (
    "Table 1. Characteristics of children with an uncomplicated in-hospital course (maximal coronary Z score below 2 on every "
    "in-hospital report), by whether an echocardiogram was done 1–30 days after discharge. Validation children had at least 30 days "
    "of potential follow-up. Laboratory values are the first (admission) and last (discharge) results of the index admission. IVIG "
    "resistance: a second IVIG course, a systemic glucocorticoid started more than 24 hours after the first IVIG, or infliximab. "
    "Risk factors: age <12 months, IVIG resistance, discharge CRP >20 mg/L or WBC >12 × 10⁹/L (in-hospital dilation only does not occur "
    "after an uncomplicated course). Fever days at admission were parsed from the admission record; data availability: "
    + '; '.join(f"{k.split(',')[0]} {', '.join(v)}" for k, v in low.items()) +
    ". SMD, absolute standardized mean difference between children with and without the echocardiogram. ALT, alanine "
    "aminotransferase; CRP, C-reactive protein; IVIG, intravenous immunoglobulin; WBC, white blood cell count.")

## 8. Figure 1 — study flow (A) and timing of the post-discharge echocardiograms (B)
Panel A uses the cohort counts of section 1 (asserted against the locks). Panel B is descriptive: the dates of the classifiable
echocardiograms after discharge of derivation children with an uncomplicated course (echocardiograms to February 28, 2025, as in
the locked analysis). The first one 1–30 days after discharge is asserted against the locked count, median and IQR.

In [ ]:
def fmt_date(d, long=False):
    """American style: 'Mar 8, 2025' in the flow chart, 'March 8, 2025' in the legend."""
    return f'{d.strftime("%B" if long else "%b")} {d.day}, {d.year}'

D, Vf = FLOW_D, FLOW_V
MAIN_D = [
    f"First inpatient admission for\nKawasaki disease, 2012–2026\nn = {nfmt(D['all_index'])}",
    f"Index admissions 2018–2024\n(2021 excluded), n = {nfmt(D['primary'])}",
    f"≥1 classifiable in-hospital\nechocardiogram, n = {nfmt(D['c1'])}",
    f"Uncomplicated course\n(Z < 2 on every report)\nn = {nfmt(D['uncomplicated'])}",
    f"First echocardiogram\n1–30 d after discharge\nn = {nfmt(D['q1'])}",
    f"Normal through 30 d;\nechocardiogram 31–90 d\nn = {nfmt(D['w2'])}",
    f"Normal through 90 d;\nechocardiogram 91–365 d\nn = {nfmt(D['w3'])}",
    f"Normal through 365 d;\nechocardiogram >365 d\nn = {nfmt(D['w4'])}"]
SIDE_D = [
    f"Excluded\nbefore 2018, n = {nfmt(D['pre2018'])}\n2021, n = {nfmt(D['y2021'])}\n2025–2026, n = {nfmt(D['y2025_26'])}",
    f"No classifiable\nin-hospital report\nn = {nfmt(D['no_inhosp_echo'])}",
    f"Coronary involvement\nin hospital (Z ≥ 2)\nn = {nfmt(D['inhosp_involvement'])}",
    f"No echocardiogram\n1–30 d after discharge\nn = {nfmt(D['no_q1'])}",
    f"Finding at 1–30 d, n = {D['q1_to_w2_finding']}\nNo echocardiogram\n31–90 d, n = {nfmt(D['q1_to_w2_noecho'])}",
    f"Finding at 31–90 d, n = {D['w2_to_w3_finding']}\nNo echocardiogram\n91–365 d, n = {nfmt(D['w2_to_w3_noecho'])}",
    f"Finding at 91–365 d, n = {D['w3_to_w4_finding']}\nNo echocardiogram\n>365 d, n = {nfmt(D['w3_to_w4_noecho'])}"]
MAIN_V = [None,
    f"First in-hospital echocardiogram\n{fmt_date(Vf['start'])}–{fmt_date(Vf['end'])}\nn = {nfmt(Vf['cand'])}",
    f"≥1 classifiable in-hospital\nechocardiogram, n = {nfmt(Vf['v'])}",
    f"Uncomplicated course\n(Z < 2 on every report)\nn = {nfmt(Vf['uncomplicated'])}",
    f"First echocardiogram\n1–30 d after discharge\nn = {nfmt(Vf['q1'])}",
    f"Normal through 30 d;\nechocardiogram 31–90 d\nn = {nfmt(Vf['w2'])}",
    f"Normal through 90 d;\nechocardiogram 91–365 d\nn = {nfmt(Vf['w3'])}",
    None]
SIDE_V = [None,
    f"No classifiable\nin-hospital report\nn = {Vf['no_class']}",
    f"Coronary involvement\nin hospital (Z ≥ 2)\nn = {nfmt(Vf['inhosp_involvement'])}",
    f"Discharged <30 d before\nthe data end, n = {Vf['short_fu']}\nNo echocardiogram\n1–30 d, n = {Vf['no_q1']}",
    f"Finding at 1–30 d, n = {Vf['q1_to_w2_finding']}\nFollow-up <90 d, n = {Vf['q1_to_w2_short']}\nNo echocardiogram\n31–90 d, n = {Vf['q1_to_w2_noecho']}",
    f"Finding at 31–90 d, n = {Vf['w2_to_w3_finding']}\nFollow-up <365 d, n = {Vf['w2_to_w3_short']}\nNo echocardiogram\n91–365 d, n = {Vf['w2_to_w3_noecho']}",
    None]

# ---- legend numbers: 2025-2026 index admissions by date of the first in-hospital echocardiogram; 1-30-d findings at the first
# echocardiogram versus a later one in the window (the flow counts any echocardiogram in the window)
v_ = MI[MI['index_year'] >= 2025]
V_SPLIT = dict(total=len(v_), cand=int((v_['first_inh'] >= EXPORT_START).sum()), earlier=int((v_['first_inh'] < EXPORT_START).sum()),
               none=int(v_['first_inh'].isna().sum()))
assert V_SPLIT['total'] == FLOW_D['y2025_26'] and V_SPLIT['cand'] == Vf['cand'] == len(CAND)
assert V_SPLIT['cand'] + V_SPLIT['earlier'] + V_SPLIT['none'] == V_SPLIT['total']
W1_FIRST = int((Q_D['post30_cat_first'] >= 1).sum()); W1_ANY = FLOW_D['q1_to_w2_finding']
assert W1_FIRST == DER[('1–30', 'involvement')][0]
EXTRA['figure1_flow'] = dict(V_SPLIT, w1_finding_first_echo=W1_FIRST, w1_finding_later_echo=W1_ANY - W1_FIRST)

# ---- panel B data: classifiable post-discharge echocardiograms of uncomplicated derivation children (one per day)
DISD = MI.set_index('research_patient_id')['dis']
TP = E[E['aha_best'].notna() & (E['exam_date'] <= ERA_END) & E['phase'].isin(POST) & E['research_patient_id'].isin(U_D['research_patient_id'])]
TP = TP[['research_patient_id', 'exam_date']].drop_duplicates()
TP['d'] = (TP['exam_date'] - DISD.reindex(TP['research_patient_id']).values).dt.days
TP = TP.sort_values(['research_patient_id', 'd'], kind='mergesort')
TP['k'] = TP.groupby('research_patient_id').cumcount() + 1
assert TP['research_patient_id'].nunique() == len(TD), (TP['research_patient_id'].nunique(), len(TD))
f30 = TP.loc[(TP['k'] == 1) & (TP['d'] <= 30), 'd']
assert len(f30) == FLOW_D['q1']
assert_lock(f"with a first echo ≤30 d n = {nfmt(len(f30))} (median day {f30.median():.0f} after discharge [IQR {f30.quantile(0.25):.0f}–{f30.quantile(0.75):.0f}]")
TIMING = []
for k in [1, 2, 3, 4]:
    x = TP.loc[TP['k'] == k, 'd'].astype(float)
    TIMING.append(dict(k=k, n=len(x), p10=x.quantile(0.10), q1=x.quantile(0.25), med=x.median(), q3=x.quantile(0.75), p90=x.quantile(0.90), dmin=x.min()))
EXTRA['figure1b_timing'] = [{kk: (round(float(v), 1) if kk not in ('k', 'n') else int(v)) for kk, v in t.items()} for t in TIMING]
print('timing of the 1st-4th post-discharge echocardiograms:', EXTRA['figure1b_timing'])

# ---- layout: flow on top (inches as data units), timing panel below
BOX_H, PITCH, TOP, FS_MAIN, FS_SIDE = 0.46, 0.70, 0.30, 7.6, 7.0
H1 = TOP + 0.35 + PITCH * (len(MAIN_D) - 1) + BOX_H / 2 + 0.05
HB = 2.35
H = H1 + HB
fig = plt.figure(figsize=(FIG_W, H))
# the flow axes keeps inches as data units and stays PAD_IN inside the left and right edges (so the saved figure is at most 7.0 in wide)
ax = fig.add_axes([PAD_IN / FIG_W, HB / H, 1 - 2 * PAD_IN / FIG_W, H1 / H]); ax.set_xlim(PAD_IN, FIG_W - PAD_IN); ax.set_ylim(0, H1); ax.axis('off')

BOXES = []
def draw_box(x, yc, w, h, text, fs, edge=BLACK, fill=WHITE):
    BOXES.append((x, yc - h / 2, x + w, yc + h / 2))
    ax.add_patch(FancyBboxPatch((x, yc - h / 2), w, h, boxstyle='round,pad=0,rounding_size=0.04', linewidth=1.0, edgecolor=edge, facecolor=fill))
    ax.text(x + w / 2, yc, text, ha='center', va='center', fontsize=fs, color=INK, linespacing=1.12)

def arrow(x0, y0, x1, y1, color):
    ax.annotate('', xy=(x1, y1), xytext=(x0, y0), arrowprops=dict(arrowstyle='-|>', lw=1.0, color=color, mutation_scale=9, shrinkA=0, shrinkB=0))

def column(x0, mains, sides, header, color):
    w_main, gap, w_side = 1.85, 0.08, 1.40
    cx = x0 + w_main / 2
    ax.text(cx, H1 - 0.20, header, ha='center', va='center', fontsize=10, fontweight='bold', color=color)
    ys = [H1 - TOP - 0.35 - i * PITCH for i in range(len(mains))]
    for i, (t, y) in enumerate(zip(mains, ys)):
        if t is None:
            continue
        draw_box(x0, y, w_main, BOX_H, t, FS_MAIN, edge=color, fill=tint(color, 0.88))
        if i + 1 < len(mains) and mains[i + 1] is not None:
            arrow(cx, y - BOX_H / 2, cx, ys[i + 1] + BOX_H / 2, color)
            if i < len(sides) and sides[i] is not None:
                ym = (y - BOX_H / 2 + ys[i + 1] + BOX_H / 2) / 2
                nl = sides[i].count('\n') + 1
                hs = 0.112 * nl + 0.07
                draw_box(x0 + w_main + gap, ym, w_side, hs, sides[i], FS_SIDE, edge=tint(color, 0.35), fill=WHITE)
                ax.plot([cx, x0 + w_main + gap], [ym, ym], color=color, lw=1.0)

column(0.08, MAIN_D, SIDE_D, 'Derivation cohort', C_DER)
column(3.56, MAIN_V, SIDE_V, 'Temporal validation cohort', C_VAL)
for i in range(len(BOXES)):                        # no two boxes of the flow touch, and all lie inside the drawing area
    a = BOXES[i]; assert PAD_IN <= a[0] and a[2] <= FIG_W - PAD_IN, a
    for b in BOXES[i + 1:]:
        assert min(a[2], b[2]) - max(a[0], b[0]) <= 0.02 or min(a[3], b[3]) - max(a[1], b[1]) <= 0.02, (a, b)
fig.text(PAD_IN / FIG_W, 1 - 0.06 / H, 'A', fontsize=14, fontweight='bold', ha='left', va='top')

# ---- panel B
axB = fig.add_axes([0.20, 0.42 / H, 0.55, 1.45 / H])
ys = [4, 3, 2, 1]
for t, y in zip(TIMING, ys):
    axB.plot([max(t['p10'], 1), t['p90']], [y, y], color=C_DER, lw=1.1, zorder=2, solid_capstyle='butt')
    axB.add_patch(Rectangle((max(t['q1'], 1), y - 0.26), t['q3'] - max(t['q1'], 1), 0.52, facecolor=tint(C_DER, 0.55), edgecolor=C_DER, lw=0.9, zorder=3))
    axB.plot([max(t['med'], 1)] * 2, [y - 0.26, y + 0.26], color='#00395A', lw=2.0, zorder=4, solid_capstyle='butt')
    axB.text(1.03, y, f"{nfmt(t['n'])}", transform=axB.get_yaxis_transform(), ha='left', va='center', fontsize=8.5, color=INK)
    axB.text(1.17, y, f"{t['med']:.0f} ({t['q1']:.0f}–{t['q3']:.0f})", transform=axB.get_yaxis_transform(), ha='left', va='center', fontsize=8.5, color=INK)
axB.text(1.03, 4.62, 'Children', transform=axB.get_yaxis_transform(), ha='left', va='bottom', fontsize=8.5, fontweight='bold')
axB.text(1.17, 4.62, 'Median (IQR), d', transform=axB.get_yaxis_transform(), ha='left', va='bottom', fontsize=8.5, fontweight='bold')
SCHED = [7, 30, 91, 182, 365]
for s in SCHED:
    axB.axvline(s, color=MGREY, lw=0.8, ls='--', zorder=1)
axB.set_xscale('log'); axB.set_xlim(1, 2500)
axB.set_xticks([1, 7, 30, 91, 182, 365, 730, 1460]); axB.set_xticklabels(['1', '7', '30', '91', '182', '365', '730', '1,460'])
axB.minorticks_off()
sec = axB.secondary_xaxis('top'); sec.set_xticks(SCHED); sec.set_xticklabels(['1 wk', '1 mo', '3 mo', '6 mo', '12 mo'], fontsize=8)
sec.tick_params(length=2.5, pad=1.5); sec.minorticks_off()
axB.set_yticks(ys); axB.set_yticklabels(['1st', '2nd', '3rd', '4th']); axB.set_ylim(0.4, 4.6)
axB.set_xlabel('Days after discharge'); axB.set_ylabel('Echocardiogram')
letter_at(fig, axB, 'B', pad_in=0.08)
save_checked(fig, 'Figure1_study_flow_and_follow_up')

LEGENDS['Figure 1'] = (
    "Figure 1. Study flow and timing of follow-up echocardiography. A, study flow. Left (blue), derivation cohort (index admissions "
    "2018–2024; 2021 was excluded because in-hospital echocardiogram reports were missing from the export). Right (red), temporal "
    f"validation cohort: of the {nfmt(V_SPLIT['total'])} index admissions in 2025–2026, the {nfmt(V_SPLIT['cand'])} whose first "
    f"in-hospital echocardiogram was on or after {fmt_date(Vf['start'], True)} (data to {fmt_date(Vf['end'], True)}); "
    f"{V_SPLIT['earlier']} had their first in-hospital echocardiogram earlier and {V_SPLIT['none']} had none. Uncomplicated course: "
    "maximal coronary Z score below 2 on every in-hospital echocardiogram report. Each later step includes children in whom every "
    "echocardiogram in the earlier windows was normal (Z < 2) and who had an echocardiogram in the next window; windows are counted "
    "from discharge. Finding: coronary involvement (Z ≥ 2) on any echocardiogram in the window (derivation, 1–30 days: "
    f"{W1_FIRST} children at the first echocardiogram and {W1_ANY - W1_FIRST} at a later one). Validation children were required to "
    "have 30, 90, or 365 days of potential follow-up for the respective windows; the window after 365 days was not assessed in the "
    "validation cohort. B, timing of the first to fourth echocardiograms after discharge in children of the derivation cohort with an "
    f"uncomplicated in-hospital course and at least 1 echocardiogram after discharge (n = {nfmt(TIMING[0]['n'])}; echocardiograms to "
    "February 28, 2025; several reports on one day counted once). Boxes, interquartile range; dark line, median; horizontal lines, 10th "
    "to 90th percentiles; logarithmic axis. Dashed lines, visits scheduled about 1 week and 1, 3, 6, and 12 months after discharge "
    "(labels at the top). Right, children with that echocardiogram and the median (IQR) number of days after discharge.")
print(LEGENDS['Figure 1'])

## 9. Figure 3 — robustness of the first-visit yield (A), measurement noise (B) and the course of first-visit aneurysms (C)
Panel A gathers every alternative estimate of the aneurysm yield at the first post-discharge echocardiogram from the locked tables
(v4 Table S2, v4.1 modules M1a, M2a and M2b, v5 section V4 and V5); each value is asserted against the locked Results text.
Panel B is the noise model of module M2b; panel C is the descriptive cut of section 4.

In [ ]:
S2T = pd.read_csv(V41 / 'TableS2_sensitivity.csv').reset_index(drop=True)
M1A = pd.read_csv(V41 / 'TableM1a_2021_sensitivity.csv')
V52 = pd.read_csv(V5 / 'TableV5_2_P1_sensitivity.csv').reset_index(drop=True)
CD = pd.read_csv(V41 / 'TableM2b_confirmed_definitions.csv').reset_index(drop=True)
NM = pd.read_csv(V41 / 'TableM2b_noise_model.csv')
RP = pd.read_csv(V41 / 'TableM2b_repeatability.csv')
M2A = KN41['MOD']['M2a']

def rt(r):
    return dict(k=int(r['events']), n=int(r['n']), pct=float(r['pct']), lo=float(r['ci_low']), hi=float(r['ci_high']))
FOREST = []
def add(group, label, d, cohort='der'):
    FOREST.append(dict(group=group, label=label, cohort=cohort, **d))

for i, pre in [(0, 'Primary'), (1, 'S1:'), (2, 'S2:'), (3, 'S3:'), (4, 'S4:'), (5, 'S6a:'), (6, 'S6b:'), (7, 'S7a:'), (9, 'S5:'), (10, 'Plan B')]:
    assert S2T.loc[i, 'analysis'].startswith(pre), (i, S2T.loc[i, 'analysis'])
    assert_lock(f"{int(S2T.loc[i, 'events'])}/{int(S2T.loc[i, 'n'])} ({S2T.loc[i, 'pct']:.2f}%)")
add('Primary analysis', 'Uncomplicated course, first echo 1–30 d', rt(S2T.loc[0]))
G = 'Cohort, timing and treatment'
for i, lab in [(1, 'Last in-hospital report Z < 2'), (2, 'First echo within 60 d'), (3, 'IVIG-treated children only'),
               (4, 'Readmissions within 30 d excluded'), (7, 'First echo on illness day 11–21')]:
    add(G, lab, rt(S2T.loc[i]))
G = 'Calendar period and data source'
add(G, 'Index admissions 2018–2020', rt(S2T.loc[5])); add(G, 'Index admissions 2022–2024', rt(S2T.loc[6]))
m1 = M1A[M1A['outcome'].str.startswith('aneurysm')].set_index('analysis')
for key, lab in [('Rule-based baseline, 2021 only', '2021, rule-based baseline'),
                 ('Pooled 2018–2024 including 2021 (AHA baseline + 2021 rule-based)', '2018–2024 including 2021')]:
    d = rt(m1.loc[key]); assert_lock(f"{d['k']}/{nfmt(d['n'])} = {d['pct']:.2f}% (95% CI {d['lo']:.2f}–{d['hi']:.2f})"); add(G, lab, d)
add(G, 'Events from exact numbers only', rt(S2T.loc[9])); add(G, 'All data, 2018–2026', rt(S2T.loc[10]))
G = 'Children without a post-discharge echo'
assert_lock(f"IPW {M2A['ipw']['pct']:.2f}% (bootstrap 95% CI {M2A['ipw']['ci_low']:.2f}–{M2A['ipw']['ci_high']:.2f})",
            f"standardized to the risk-factor distribution {M2A['standardized']['pct']:.2f}% ({M2A['standardized']['ci_low']:.2f}–{M2A['standardized']['ci_high']:.2f})",
            f"lower bound {M2A['lower_bound']['events']}/{nfmt(M2A['lower_bound']['n'])} = {M2A['lower_bound']['pct']:.2f}%")
add(G, 'Inverse probability weighting', dict(k=None, n=None, pct=M2A['ipw']['pct'], lo=M2A['ipw']['ci_low'], hi=M2A['ipw']['ci_high']))
add(G, 'Standardized to all children', dict(k=None, n=None, pct=M2A['standardized']['pct'], lo=M2A['standardized']['ci_low'], hi=M2A['standardized']['ci_high']))
lb_ = M2A['lower_bound']
add(G, 'None of them with an aneurysm', dict(k=lb_['events'], n=lb_['n'], pct=lb_['pct'], lo=lb_['ci_low'], hi=lb_['ci_high']))
for i in [1, 2, 3, 4, 5, 6]:
    d = rt(CD.loc[i]); assert_lock(f"{d['k']}/{nfmt(d['n'])} = {d['pct']:.2f}% ({d['lo']:.2f}–{d['hi']:.2f})")
G = 'Stricter event definitions'
add(G, 'Z ≥ 2.5 again within 90 d', rt(CD.loc[1])); add(G, 'Confirmed or not reimaged ≤90 d', rt(CD.loc[2])); add(G, 'Exact maximal Z ≥ 3.0', rt(CD.loc[3]))
G = 'Exact in-hospital maximum (n = 276)'
add(G, 'Z ≥ 2.5, primary definition', rt(CD.loc[6])); add(G, 'Rise ≥ repeatability coefficient, SD', rt(CD.loc[4]))
add(G, 'Rise ≥ repeatability coefficient, robust', rt(CD.loc[5]))
G = 'Temporal validation'
for i, lab in [(0, 'Primary analysis'), (1, 'Without the conclusion rule'), (2, 'Derivation-era categories')]:
    d = rt(V52.loc[i]); assert_lock(f"{d['k']}/{d['n']} = {d['pct']:.2f}% (95% CI {d['lo']:.2f}–{d['hi']:.2f})"); add(G, lab, d, 'val')
assert_lock('Confirmed new aneurysms (Z ≥ 2.5 again on the next echo ≤90 d; definition (a) of v4.1 module M2b): v5 6/615 = 0.98% (95% CI 0.45–2.11)')
add(G, 'Z ≥ 2.5 again within 90 d', dict(k=6, n=615, pct=0.98, lo=0.45, hi=2.11), 'val')
for r in FOREST:
    if r['k'] is not None:                              # Wilson intervals recomputed from the counts agree with the locked ones
        lo, hi = wilson(r['k'], r['n']); assert abs(lo - r['lo']) < 0.006 and abs(hi - r['hi']) < 0.006, r
PRIMARY = FOREST[0]['pct']

# ---- layout
lines = []
for r in FOREST:
    if not lines or lines[-1][1]['group'] != r['group']:
        lines.append(('h', r))
    lines.append(('r', r))
nl = len(lines); LINE_IN = 0.152
HA = nl * LINE_IN + 0.75                 # panel A height incl. x axis and the column headings
HBC = 2.98
H = HA + HBC + 0.15
fig = plt.figure(figsize=(FIG_W, H))
axA = fig.add_axes([0.40, (HBC + 0.15 + 0.42) / H, 0.37, (nl * LINE_IN) / H])
bt = mtransforms.blended_transform_factory(fig.transFigure, axA.transData)
for i, (typ, r) in enumerate(lines):
    y = nl - 1 - i
    col = C_VAL if r['cohort'] == 'val' else C_DER
    if typ == 'h':
        axA.text(0.012, y, r['group'], transform=bt, ha='left', va='center', fontsize=8.5, fontweight='bold', color=C_VAL if r['cohort'] == 'val' else INK)
        continue
    axA.errorbar(r['pct'], y, xerr=[[max(r['pct'] - r['lo'], 0)], [max(r['hi'] - r['pct'], 0)]], fmt='o', ms=4.6, color=col, ecolor=col,
                 elinewidth=1.0, capsize=1.8, zorder=3)
    axA.text(0.032, y, r['label'], transform=bt, ha='left', va='center', fontsize=8, color=INK)
    axA.text(1.03, y, '—' if r['k'] is None else f"{r['k']}/{nfmt(r['n'])}", transform=axA.get_yaxis_transform(), ha='left', va='center', fontsize=8)
    axA.text(1.21, y, f"{r['pct']:.2f} ({r['lo']:.2f}–{r['hi']:.2f})", transform=axA.get_yaxis_transform(), ha='left', va='center', fontsize=8)
axA.text(1.03, nl - 0.35, 'n/N', transform=axA.get_yaxis_transform(), ha='left', va='bottom', fontsize=8, fontweight='bold')
axA.text(1.21, nl - 0.35, '% (95% CI)', transform=axA.get_yaxis_transform(), ha='left', va='bottom', fontsize=8, fontweight='bold')
axA.axvline(PRIMARY, color=MGREY, lw=0.8, ls='--', zorder=1)
axA.set_ylim(-0.7, nl - 0.3); axA.set_yticks([]); axA.spines['left'].set_visible(False)
axA.set_xlim(0, 6); axA.set_xticks(range(0, 7)); axA.set_xlabel('Aneurysm at the first echo after discharge (%)')
fig.text(PAD_IN / FIG_W, 1 - 0.04 / H, 'A', fontsize=14, fontweight='bold', ha='left', va='top')

# ---- B: noise model
axB = fig.add_axes([1.55 / FIG_W, 1.05 / H, 1.60 / FIG_W, 1.45 / H])
NML = ['SD, later pairs', 'SD, in-hospital pairs', 'Robust, later pairs', 'Robust, in-hospital pairs']
obs = int(NM['observed_aneurysms'].iloc[0])
for j, r in NM.reset_index(drop=True).iterrows():
    y = len(NM) - 1 - j
    axB.barh(y, r['expected_from_noise'], height=0.55, color=tint(C_DER, 0.5), edgecolor='none', zorder=2)
    axB.text(max(r['expected_from_noise'], obs) + 2.0, y, f"{r['expected_from_noise']:.1f}", va='center', fontsize=8)
axB.axvline(obs, color=INK, lw=1.0, ls='--', zorder=3)
axB.set_yticks(range(len(NM))); axB.set_yticklabels(NML[::-1], fontsize=8.5); axB.set_xlim(0, 50); axB.set_ylim(-0.6, len(NM) - 0.4)
axB.set_xlabel('Expected aneurysms (n)')

# ---- C: AHA category at detection and at the last report after day 30 (first-visit aneurysms)
CAT_STYLE = [(0, 'No involvement', '#F2F2F2', None), (1, 'Dilation only', NEJM['yellow'], None), (2, 'Small aneurysm', NEJM['orange'], None),
             (3, 'Medium aneurysm', NEJM['red'], None), (4, 'Large or giant', '#6E1F15', None), (-1, 'No report after 30 d', WHITE, '////')]
DARK_FILL = {NEJM['red'], '#6E1F15'}
axC = fig.add_axes([4.75 / FIG_W, 1.05 / H, 1.70 / FIG_W, 1.45 / H])
rows = [('Derivation, detection', det_D, len(na_ids)), ('Derivation, last report', last_D, len(na_ids)),
        ('Validation, detection', det_V, len(V54)), ('Validation, last report', last_V, len(V54))]
ysC = [3.3, 2.5, 1.0, 0.2]
for (lab, cnt, tot), y in zip(rows, ysC):
    left = 0.0
    for code, name, color, hatch in CAT_STYLE:
        k = int(cnt.get(code, 0))
        if k == 0:
            continue
        w = k / tot * 100
        axC.barh(y, w, left=left, height=0.6, color=color, edgecolor=MGREY if (hatch or code == 0) else WHITE, linewidth=0.8, hatch=hatch, zorder=2)
        if not hatch and w / 100 * axC.get_position().width * FIG_W >= 0.16:      # count inside the segment
            axC.text(left + w / 2, y, str(k), ha='center', va='center', fontsize=8, color=WHITE if color in DARK_FILL else INK,
                     bbox=dict(boxstyle='square,pad=0.06', facecolor=WHITE, edgecolor='none') if hatch else None, zorder=4)
        else:                                                         # narrow or hatched segment: count just above or below the bar
            up = 'detection' in lab
            axC.text(left + w / 2, y + (0.36 if up else -0.36), str(k), ha='center', va='bottom' if up else 'top', fontsize=8, color=INK)
        left += w
    axC.text(103, y, f'n = {tot}', va='center', fontsize=8)
axC.set_yticks(ysC); axC.set_yticklabels([r[0] for r in rows], fontsize=8.5)
for tl, r in zip(axC.get_yticklabels(), rows):
    tl.set_color(C_DER if r[0].startswith('Derivation') else C_VAL)
axC.set_xlim(0, 100); axC.set_ylim(-0.45, 3.75); axC.set_xlabel('Children (%)')
used = set().union(*[{c for c, v in cnt.items() if v} for _, cnt, _ in rows])
shown = [st for st in CAT_STYLE if st[0] in used]
handles = [Rectangle((0, 0), 1, 1, facecolor=c, edgecolor=MGREY if (h_ or code == 0) else c, linewidth=0.8, hatch=h_) for code, _, c, h_ in shown]
fig.canvas.draw(); y_lab = axC.xaxis.label.get_window_extent(fig.canvas.get_renderer()).y0 / fig.bbox.height
fig.legend(handles, [n_ for _, n_, _, _ in shown], loc='upper right', bbox_to_anchor=((FIG_W - PAD_IN - 0.02) / FIG_W, y_lab - 0.04 / H), ncol=2,
           fontsize=8, handlelength=1.2, columnspacing=0.9, handletextpad=0.4, frameon=False, borderaxespad=0)
letter_at(fig, axB, 'B', x_from='ticks')
letter_at(fig, axC, 'C', x_from='ticks')
save_checked(fig, 'Figure3_robustness_and_course_of_first_visit_aneurysms')

fa_ = {r['label']: r for r in FOREST}
LEGENDS['Figure 3'] = (
    "Figure 3. Robustness of the aneurysm yield at the first echocardiogram after discharge, and the course of the aneurysms found. "
    "A, aneurysm (maximal Z ≥ 2.5) at the first echocardiogram 1–30 days after an uncomplicated in-hospital course in the primary "
    "analysis and in alternative analyses (blue, derivation; red, temporal validation). Whiskers, 95% CIs (Wilson; bootstrap for the "
    "weighted and standardized estimates); dashed line, the primary estimate. Alternative analyses: baseline defined by the last "
    "in-hospital report only; first echocardiogram within 60 days; children treated with IVIG; readmissions within 30 days excluded; "
    "first echocardiogram on illness days 11–21; index admissions 2018–2020 or 2022–2024; 2021 with a rule-based in-hospital baseline "
    "from the discharge record, alone and pooled with 2018–2024; events counted only from reports with exact numbers; all index years "
    "2018–2026 (except 2021) and all echocardiograms, with the wording model for reports without numbers; the 1,221 children without an "
    "echocardiogram 1–30 days after discharge handled by inverse probability weighting, by standardization to the risk-factor "
    "distribution of all 2,839 children, or by assuming that none had an aneurysm; stricter event definitions (Z ≥ 2.5 again on the next "
    "echocardiogram within 90 days, also counting children not reimaged; exact maximal Z ≥ 3.0); in the 276 children whose in-hospital "
    "maximum was an exact Z score, a rise over that maximum at least as large as the repeatability coefficient (SD-based, prespecified; "
    "robust, post hoc); and, in the validation cohort, the category assigned without the conclusion rule for non-proximal segments or "
    "with the derivation-era classification. B, aneurysms expected from measurement noise alone among the 276 children whose in-hospital "
    "maximum was an exact Z score, if each child's true Z had stayed at that maximum, under 4 estimates of the within-child SD from "
    "consecutive reports more than 30 days after discharge (later pairs) or 1–3 days apart in hospital (in-hospital pairs); the "
    "SD-based estimates were prespecified and the robust estimates (median absolute deviation) were added post hoc because the "
    f"differences were heavy-tailed. Dashed line, the {obs} aneurysms observed; probability of at least {obs} under noise alone, in the "
    "order shown: " + ', '.join(f"{p:.2f}" if p >= 0.1 else f"{p:.3f}" for p in NM['p_observed_or_more_under_noise_only']) +
    ". C, aneurysms found at the first echocardiogram 1–30 days after an uncomplicated in-hospital course: AHA category at detection "
    f"and at the last report more than 30 days after discharge (derivation, median {last_day_D.median():.0f} days after discharge; "
    "echocardiograms to February 28, 2025). Medium or larger aneurysm at any later echocardiogram in the derivation cohort: "
    f"{fate['later_max_medium_plus']} children. No involvement, Z < 2; small aneurysm, Z 2.5 to <5; medium, Z 5 to <10; large or giant, "
    "Z ≥ 10 or internal diameter ≥ 8 mm; no report after 30 d, no echocardiogram more than 30 days after discharge. AHA, American Heart "
    "Association; echo, echocardiogram; IVIG, intravenous immunoglobulin.")
print(LEGENDS['Figure 3'])

## 10. Figure 4 — timing of coronary findings (A–C) and course of in-hospital aneurysms by size (D)
A and B use the locked Table S3 of v4.1 (coronary status by illness day). C uses the locked Table 3a (section 4). D is recomputed from
the frozen per-report table with the definitions of the locked analysis R4 (time 0 = discharge; event = first report below Z 2.5;
censored at the last report and at 1,095 days; echocardiograms to February 28, 2025); every estimate printed in the locks is asserted.

In [ ]:
S3T = pd.read_csv(V41 / 'TableS3_status_by_illness_day.csv')
s3c = S3T[(S3T['scope'] == 'primary') & (S3T['population'] == 'Cohort C1')].reset_index(drop=True)
s3u = S3T[(S3T['scope'] == 'primary') & (S3T['population'] == 'Uncomplicated in hospital (Z<2)')].reset_index(drop=True)
assert list(s3c['illness_day']) == list(s3u['illness_day'])
for _, r in s3c.iterrows():
    assert_lock(f"{r['illness_day']} {r['involvement_pct']:.1f} (aneurysm {r['aneurysm_pct']:.1f}; n = {nfmt(r['children'])})")
assert_lock('First documented involvement (n = 440) by illness day 10 72.0%, 14 86.4%, 21 94.8%, 42 97.5%',
            'aneurysm (n = 331) by day 10 66.5%, 14 83.7%, 21 94.6%, 42 98.2%, 90 98.8%')
WINS = list(s3c['illness_day'])
SEV3 = [('involvement_pct', 'Any involvement (Z ≥ 2)', tint(C_DER, 0.55), 'o'), ('aneurysm_pct', 'Aneurysm (Z ≥ 2.5)', C_DER, 's'),
        ('medium_plus_pct', 'Medium or larger (Z ≥ 5)', '#00395A', '^')]

# D: course of in-hospital aneurysms by size (locked R4)
SIZES = [(2, 'Small', NEJM['orange']), (3, 'Medium', NEJM['red']), (4, 'Large or giant', '#6E1F15')]
T6 = pd.read_csv(V41 / 'Table6_time_to_event.csv')
EPOST = E[E['aha_best'].notna() & (E['exam_date'] <= ERA_END) & E['phase'].isin(POST)]
DISS = MI.set_index('research_patient_id')['dis']
REG, NORM = {}, {}
for code, lab, _ in SIZES:
    ids = C1.loc[C1['in_max'] == code, 'research_patient_id']
    REG[code] = tte_below(EPOST[EPOST['research_patient_id'].isin(ids)], 'aha_best', DISS, 1)
    NORM[code] = tte_below(EPOST[EPOST['research_patient_id'].isin(ids)], 'aha_best', DISS, 0)
    kr, kn = km(REG[code]['t'], REG[code]['e']), km(NORM[code]['t'], NORM[code]['e'])
    name = 'Large/giant' if code == 4 else lab
    assert_lock(f"{name} (n = {len(REG[code])}): below Z 2.5 by 90 d {ci_lock(kr, 90)}, 1 y {ci_lock(kr, 365)}, 2 y {ci_lock(kr, 730)}; "
                f"normal lumen (Z < 2) by 1 y {ci_lock(kn, 365)}, 2 y {ci_lock(kn, 730)}")
    t6 = T6[(T6['scope'] == 'primary') & T6['analysis'].str.startswith('TTE-2 regression') & (T6['group'] == name)].set_index('time_d')
    for q in [90, 365, 730]:
        s_, l_, h_ = km_at(kr, q)
        assert abs(round((1 - s_) * 100, 2) - t6.loc[q, 'cum_pct']) < 0.006 and abs(round((1 - h_) * 100, 2) - t6.loc[q, 'ci_low']) < 0.006
lr_size = logrank(pd.concat([REG[c]['t'] for c, _, _ in SIZES]), pd.concat([REG[c]['e'] for c, _, _ in SIZES]),
                  np.concatenate([[c] * len(REG[c]) for c, _, _ in SIZES]))
assert lr_size[2] < 0.001                                              # locked: log-rank P < 0.001
KMR = {c: km(REG[c]['t'], REG[c]['e']) for c, _, _ in SIZES}
KMN = {c: km(NORM[c]['t'], NORM[c]['e']) for c, _, _ in SIZES}

# ---- layout (inches)
W = FIG_W
L1, R1 = 1.10, 6.72                      # A and B
HA, GAB, HB, UB = 1.65, 0.34, 1.20, 0.98  # A, gap, B, space under B (ticks, title, numbers examined)
TOPM, GBC, HC, UC = 0.34, 0.42, 1.85, 1.02
H = TOPM + HA + GAB + HB + UB + GBC + HC + UC
fig = plt.figure(figsize=(W, H))
yC = UC; yB = UC + HC + GBC + UB; yA = yB + HB + GAB
axA = fig.add_axes([L1 / W, yA / H, (R1 - L1) / W, HA / H])
axB = fig.add_axes([L1 / W, yB / H, (R1 - L1) / W, HB / H])
axC = fig.add_axes([L1 / W, yC / H, 1.95 / W, HC / H])
axD = fig.add_axes([4.20 / W, yC / H, (R1 - 4.20) / W, HC / H])
x = np.arange(len(WINS))
for ax, tab, keys, ymax, yt, who in [(axA, s3c, SEV3, 30, range(0, 31, 10), 'All children'),
                                     (axB, s3u, SEV3[:2], 6, range(0, 7, 2), 'Uncomplicated in-hospital course')]:
    for col, lab, colr, mk in keys:
        ax.plot(x, tab[col], color=colr, marker=mk, ms=5, lw=1.4, label=lab, zorder=3, clip_on=False)
    ax.set_xticks(x); ax.set_xlim(-0.5, len(WINS) - 0.5); ax.set_ylim(0, ymax); ax.set_yticks(list(yt))
    ax.set_ylabel('Children (%)')
    ax.text(0.01, 0.97, who, transform=ax.transAxes, ha='left', va='top', fontsize=8.5, color=INK)
axA.tick_params(labelbottom=False)
axB.set_xticklabels(WINS, fontsize=8.5); axB.set_xlabel('Day of illness')
axA.legend(loc='lower right', bbox_to_anchor=(1.0, 1.0), ncol=3, fontsize=8, handlelength=1.8, handletextpad=0.45, columnspacing=1.1,
           borderaxespad=0.2)
bt = mtransforms.blended_transform_factory(axB.transData, fig.transFigure)
for i, (lab, tab) in enumerate([('No. examined', None), ('All children', s3c), ('Uncomplicated', s3u)]):
    yy = (yB - 0.56 - i * 0.17) / H
    fig.text((L1 - 0.06) / W, yy, lab, ha='right', va='center', fontsize=8, fontweight='bold' if tab is None else 'normal', color=INK)
    if tab is not None:
        for xi, n in zip(x, tab['children']):
            axB.text(xi, yy, nfmt(n), transform=bt, ha='center', va='center', fontsize=8, color=INK)

# C: window of first documentation (derivation, cohort C1)
xw = 0.36
for i, (col, lab) in enumerate(FIND):
    f = F3A[col]
    for dx, k, color, leg in [(-xw / 2, f['w1'], C_DER, '1–30 d after discharge'), (xw / 2, f['later'], tint(C_DER, 0.55), '>30 d after discharge')]:
        pct = k / f['total'] * 100; lo, hi = wilson(k, f['total'])
        axC.bar(i + dx, pct, width=xw, color=color, edgecolor='none', label=leg if i == 0 else None, zorder=2)
        axC.errorbar(i + dx, pct, yerr=[[pct - lo], [hi - pct]], fmt='none', ecolor=INK, elinewidth=0.9, capsize=2.5, capthick=0.9, zorder=3)
        axC.text(i + dx, hi + 0.7, f"{k}/{f['total']}", ha='center', va='bottom', fontsize=8, rotation=90, color=INK)
axC.set_xticks(range(len(FIND))); axC.set_xticklabels([lab for _, lab in FIND], fontsize=8.5)
axC.set_ylabel('First documented (%)'); axC.set_ylim(0, 30); axC.set_xlim(-0.6, len(FIND) - 0.4)
axC.legend(loc='upper left', ncol=1, fontsize=8, handlelength=1.1, handletextpad=0.4, borderaxespad=0.2)

# D: first report below Z 2.5 after discharge, by in-hospital size
km_panel(axD, [(lab, REG[c], col) for c, lab, col in SIZES], FU_REG, [0, 180, 365, 730, 1095], 'Below Z 2.5 (%)', (0, 100),
         range(0, 101, 20))
fig.align_ylabels([axA, axB, axC])                     # y-axis titles of the left-hand panels in one column
for ax, letter in [(axA, 'A'), (axB, 'B'), (axC, 'C'), (axD, 'D')]:
    letter_at(fig, ax, letter, pad_in=0.08 if ax is not axB else 0.05)
save_checked(fig, 'Figure4_timing_of_findings_and_course_of_aneurysms')

fa = F3A
# first documented 'in hospital' in Table 3a includes echocardiograms before the index admission: children counted there whose
# in-hospital maximum was below the finding (C1 in_max is the maximum of the index admission only)
PRE_ONLY = {col: F3A[col]['inhosp'] - int((C1['in_max'] >= thr).sum()) for col, thr in [('involvement_Z2', 1), ('aneurysm_Z2.5', 2), ('medium_or_larger_Z5', 3)]}
assert all(v >= 0 for v in PRE_ONLY.values()), PRE_ONLY
EXTRA['figure4c_before_admission'] = PRE_ONLY
pre_txt = '; '.join(f"for {v} of the {F3A[c]['inhosp']} {nm}, on an echocardiogram before admission" for (c, nm), v in
                    zip([('involvement_Z2', 'children with involvement'), ('aneurysm_Z2.5', 'aneurysms'), ('medium_or_larger_Z5', 'medium or larger aneurysms')],
                        PRE_ONLY.values()) if v)
LEGENDS['Figure 4'] = (
    "Figure 4. When coronary findings were seen, and the course of in-hospital aneurysms (derivation cohort). A and B, coronary status "
    "by day of illness: A, children with an in-hospital echocardiogram; B, children with an uncomplicated in-hospital course (Z < 2 on "
    "every in-hospital report), in whom every finding was first seen after discharge (medium or larger aneurysms are not shown because "
    "they were too few). Only children with a recorded date of fever onset contribute. Each child contributes the largest finding among "
    "the echocardiograms in each illness-day window; percentages are of the children examined in that window (numbers under the axes). "
    "Of first-documented involvement (n = 440) and aneurysms (n = 331) with a recorded onset date, 94.8% and 94.6% had been recorded by "
    "illness day 21, and 97.5% and 98.2% by day 42. C, children with each finding at any time (any involvement "
    f"n = {fa['involvement_Z2']['total']}, aneurysm n = {fa['aneurysm_Z2.5']['total']}, and medium or larger "
    f"n = {fa['medium_or_larger_Z5']['total']}) whose finding was first documented 1–30 days or more than 30 days after discharge; the "
    f"remainder were first documented in hospital ({fa['involvement_Z2']['inhosp']}, {fa['aneurysm_Z2.5']['inhosp']}, and "
    f"{fa['medium_or_larger_Z5']['inhosp']} children; {pre_txt}). Whiskers, Wilson 95% CIs; labels, children/children with the "
    "finding. D, first echocardiogram below the aneurysm threshold (maximal Z < 2.5) after discharge in children whose largest "
    f"in-hospital finding was an aneurysm and who had at least 1 echocardiogram after discharge (small n = {len(REG[2])}, medium "
    f"n = {len(REG[3])}, and large or giant n = {len(REG[4])}); 1 − Kaplan–Meier, time 0 = discharge, follow-up censored at the last "
    "echocardiogram and at 1,095 days; shaded bands, 95% CIs. At 1 year: small "
    f"{ci_lock(KMR[2], 365)}, medium {ci_lock(KMR[3], 365)}, and large or giant {ci_lock(KMR[4], 365)}; log-rank across sizes "
    f"{p_txt(lr_size[2])}. A normal lumen (maximal Z < 2) was reached by 1 year in {ci_lock(KMN[2], 365)}, {ci_lock(KMN[3], 365)}, "
    f"and {ci_lock(KMN[4], 365)}, and by 2 years in {ci_lock(KMN[2], 730)}, {ci_lock(KMN[3], 730)}, and {ci_lock(KMN[4], 730)}, "
    "respectively. Because an event is dated at the echocardiogram that showed it, early regression partly reflects when the first "
    "follow-up echocardiogram was done. Any involvement, Z ≥ 2; aneurysm, Z ≥ 2.5; medium or larger, Z ≥ 5.")
print(LEGENDS['Figure 4'])

## 11. Figure 5 — risk factors and risk-based selection
A and D: locked Table 4a, Table M3b and v5 section S7 (section 5). B: locked Table M3a (exact in-hospital Z gradient). C: locked Table
M1b (age groups). E and F: recomputed from the frozen per-report table with the definitions of the locked analysis R5 (time 0 =
discharge; event = first post-discharge report with Z ≥ 2.5; censored at the last report and at 730 days; echocardiograms to
February 28, 2025); every estimate printed in the locks is asserted.

In [ ]:
# B: exact in-hospital Z gradient below the aneurysm threshold (children discharged with Z < 2.5 whose in-hospital maximum was an exact Z)
GR = pd.read_csv(V41 / 'TableM3a_Z_gradient.csv')
GRF = pd.read_csv(V41 / 'TableM3a_Z_gradient_firth.csv')
for _, r in GR.iterrows():
    assert_lock(f"Z {r['in_hospital_exact_max_Z']} {int(r['events'])}/{int(r['n'])} ({r['pct']:.2f}%; {r['ci_low']:.2f}–{r['ci_high']:.2f})")
    lo, hi = wilson(int(r['events']), int(r['n'])); assert abs(lo - r['ci_low']) < 0.006 and abs(hi - r['ci_high']) < 0.006
assert_lock(f"n = {int(GR['n'].sum())} of 1,702", 'Cochran–Armitage trend P = 0.020')
gc, ga = GRF.iloc[0], GRF.iloc[1]
assert gc['model'] == 'crude' and ga['model'].startswith('adjusted')
OR_TXT = (f"crude {gc['OR']:.2f} ({gc['CI_low']:.2f}–{gc['CI_high']:.2f}), P = {gc['p']:.3f}",
          f"adjusted for age <12 mo and IVIG resistance {ga['OR']:.2f} ({ga['CI_low']:.2f}–{ga['CI_high']:.2f}), P = {ga['p']:.3f}")
assert_lock(*OR_TXT)
GRL = ['<1', '1–<1.5', '1.5–<2', '2–<2.5']
assert list(GR['in_hospital_exact_max_Z']) == GRL

# C: age groups (locked Table M1b)
INF = pd.read_csv(V41 / 'TableM1b_infants.csv').set_index('age_group')
AGES = ['<6 mo', '6–11 mo', '≥12 mo']
MEAS_C = [('first_echo_aneurysm_uncomplicated', 'In-hospital Z < 2', C_DER), ('first_echo_aneurysm_Z_lt_2_5', 'In-hospital Z < 2.5', tint(C_DER, 0.55))]
AGE_KN = {}
for a in AGES + ['<3 mo']:
    for c, _, _ in MEAS_C:
        assert_lock(INF.loc[a, c])                     # e.g. '7/184 (3.8%; 1.9–7.6)'
        AGE_KN[(a, c)] = parse_kn(INF.loc[a, c])

# E and F: new aneurysm after discharge by risk-factor count and after in-hospital dilation only (locked R5)
cov5 = COV.reindex(TD.index)
inflam = ((cov5['CRPc_dis'] > 20) | (cov5['WBC_dis'] > 12)).fillna(False)
rc = ((cov5['age_m'] < 12).astype(int) + cov5['ivig_resistant'].fillna(False).astype(int) + inflam.astype(int)).clip(upper=2)
TD5 = TD.assign(rc=rc.values)
RCL = {0: '0 factors', 1: '1 factor', 2: '≥2 factors'}
KMRC = {g: km(TD5.loc[TD5['rc'] == g, 't'], TD5.loc[TD5['rc'] == g, 'e']) for g in [0, 1, 2]}
for g in [0, 1, 2]:
    assert_lock(f"{RCL[g]} {ci_txt(KMRC[g], 365)}")
lr_rc = logrank(TD5['t'], TD5['e'], TD5['rc'])
DIL_IDS = C1.loc[C1['in_max'] == 1, 'research_patient_id']
TDL = tte(E[E['aha_best'].notna() & (E['exam_date'] <= ERA_END) & E['phase'].isin(POST) & E['research_patient_id'].isin(DIL_IDS)],
          'aha_best', MI.set_index('research_patient_id')['dis'])
TDL.loc[TDL['t'] > FU_NEW, 'e'] = 0; TDL['t'] = TDL['t'].clip(upper=FU_NEW)
KMDL = km(TDL['t'], TDL['e'])
assert_lock(f"Dilation only (Z 2–<2.5) {ci_txt(KMDL, 365)}", f"None (Z<2) {ci_txt(KMD, 365)}", f"in-hospital dilation only (n = {len(TDL)})")
lr_dil = logrank(pd.concat([TD['t'], TDL['t']]), pd.concat([TD['e'], TDL['e']]), np.r_[['none'] * len(TD), ['dil'] * len(TDL)])
print('log-rank, risk count:', lr_rc, '| dilation only vs none:', lr_dil)
assert p_txt(lr_rc[2]) == 'P = 0.002' and lr_dil[2] < 0.001        # locked: P = 0.002 and P < 0.001

# ---- layout (inches): three rows of two panels
W = FIG_W
XL, WL, XR, WR = 0.92, 2.25, 4.50, 2.30
TOPM, HR, GAP, HR3, U3 = 0.34, 1.70, 0.82, 1.55, 1.03
H = TOPM + HR + GAP + HR + GAP + HR3 + U3
y3 = U3; y2 = y3 + HR3 + GAP; y1 = y2 + HR + GAP
fig = plt.figure(figsize=(W, H))
axA = fig.add_axes([XL / W, y1 / H, WL / W, HR / H]); axB = fig.add_axes([XR / W, y1 / H, WR / W, HR / H])
axC = fig.add_axes([XL / W, y2 / H, WL / W, HR / H]); axD = fig.add_axes([XR / W, y2 / H, WR / W, HR / H])
axE = fig.add_axes([XL / W, y3 / H, WL / W, HR3 / H]); axF = fig.add_axes([XR / W, y3 / H, WR / W, HR3 / H])

def bars(ax, groups, series, ymax, yticks, off):
    """Grouped bars with Wilson whiskers and k/n labels; series = [(label, color, {group: (k, n)})]."""
    nb = len(series); bw = 0.72 / nb
    for j, (lab, col, kn) in enumerate(series):
        for i, g in enumerate(groups):
            k, n = kn[g]; pct = k / n * 100; lo, hi = wilson(k, n); xx = i + (j - (nb - 1) / 2) * bw
            ax.bar(xx, pct, width=bw, color=col, edgecolor='none', label=lab if i == 0 else None, zorder=2)
            ax.errorbar(xx, pct, yerr=[[pct - lo], [hi - pct]], fmt='none', ecolor=INK, elinewidth=0.9, capsize=2.2, capthick=0.9, zorder=3)
            ax.text(xx, hi + off, f'{k}/{nfmt(n)}', ha='center', va='bottom', fontsize=8, rotation=90, color=INK)
    ax.set_xticks(range(len(groups))); ax.set_xlim(-0.6, len(groups) - 0.4); ax.set_ylim(0, ymax); ax.set_yticks(yticks)

# A: aneurysm at the first post-discharge echo by number of risk factors (children discharged with Z < 2.5)
bars(axA, CNT_LAB, [('Derivation', C_DER, RF_D), ('Validation', C_VAL, RF_V)], 12, range(0, 13, 4), 0.3)
axA.set_xticklabels(CNT_LAB); axA.set_xlabel('Risk factors (n)'); axA.set_ylabel('Aneurysm (%)')
axA.legend(loc='lower right', bbox_to_anchor=(1.0, 1.0), ncol=2, fontsize=8, handlelength=1.1, handletextpad=0.4, columnspacing=0.9,
           borderaxespad=0.2)
# B: by exact in-hospital maximal Z
bars(axB, GRL, [('', C_DER, {g: (int(r['events']), int(r['n'])) for g, (_, r) in zip(GRL, GR.iterrows())})], 30, range(0, 31, 10), 0.7)
axB.set_xticklabels(GRL, fontsize=9); axB.set_xlabel('In-hospital maximal Z'); axB.set_ylabel('Aneurysm (%)')
# C: by age at admission
bars(axC, AGES, [(lab, col, {a: AGE_KN[(a, c)] for a in AGES}) for c, lab, col in MEAS_C], 12, range(0, 13, 4), 0.3)
axC.set_xticklabels(AGES); axC.set_xlabel('Age at admission'); axC.set_ylabel('Aneurysm (%)')
axC.legend(loc='lower right', bbox_to_anchor=(1.0, 1.0), ncol=2, fontsize=8, handlelength=1.1, handletextpad=0.4, columnspacing=0.9,
           borderaxespad=0.2)
# D: echocardiograms avoided and aneurysms detected by selection rule
MK = {'S0': ('s', 'All children'), 'S1': ('o', '≥1 risk factor'), 'S2': ('^', '≥2 risk factors'),
      'S3': ('D', 'Dilation only or\nIVIG resistance')}
OFF = {'S0': (5, -5, 'left', 'top'), 'S1': (-5, -5, 'right', 'top'), 'S2': (-6, -5, 'right', 'top'), 'S3': (5, 5, 'left', 'bottom')}
for s in STRAT:
    m, lab = MK[s['key']]
    axD.errorbar(s['x'], s['y'], xerr=[[s['x'] - s['xl']], [s['xh'] - s['x']]], yerr=[[s['y'] - s['yl']], [s['yh'] - s['y']]],
                 fmt='none', ecolor=tint(C_DER, 0.45), elinewidth=0.9, capsize=2, zorder=2)
    hollow = s['key'] == 'S3'                     # open diamond: it lies on the ≥2-risk-factor triangle, which stays visible inside it
    axD.plot(s['x'], s['y'], marker=m, ms=6.5 if hollow else 6, color=C_DER, markeredgecolor=C_DER, markerfacecolor='none' if hollow else C_DER,
             markeredgewidth=1.3 if hollow else 1.0, linestyle='none', label=lab, zorder=4 if hollow else 3)
    dx_, dy_, ha, va = OFF[s['key']]
    axD.annotate(f"{s['det']}/{s['tot']}", (s['x'], s['y']), xytext=(dx_, dy_), textcoords='offset points', ha=ha, va=va, fontsize=8, color=INK)
axD.errorbar(VAL_PT['x'], VAL_PT['y'], yerr=[[VAL_PT['y'] - VAL_PT['yl']], [VAL_PT['yh'] - VAL_PT['y']]], fmt='none', ecolor=tint(C_VAL, 0.45),
             elinewidth=0.9, capsize=2, zorder=2)
axD.plot(VAL_PT['x'], VAL_PT['y'], marker='o', ms=6.5, color=C_VAL, markeredgecolor=C_VAL, linestyle='none', label='≥1 risk factor, validation',
         zorder=3)
axD.annotate(f"{VAL_PT['det']}/{VAL_PT['tot']}", (VAL_PT['x'], VAL_PT['y']), xytext=(6, -5), textcoords='offset points', ha='left', va='top',
             fontsize=8, color=INK)
axD.set_xlim(-4, 100); axD.set_ylim(0, 108); axD.set_xticks(range(0, 101, 20)); axD.set_yticks(range(0, 101, 20))
axD.set_xlabel('Echocardiograms avoided (%)'); axD.set_ylabel('Sensitivity (%)')
axD.legend(loc='lower left', fontsize=8, handlelength=1.0, handletextpad=0.4, borderaxespad=0.2, labelspacing=0.3)
# E, F: cumulative incidence of a new aneurysm after discharge
RC_COL = {0: tint(C_DER, 0.45), 1: C_DER, 2: '#00395A'}           # the blue scale of Figure 4A (lightest step a little darker)
TQ5 = [0, 180, 365, 545, 730]
def band_top(kms):
    """Largest upper 95% confidence limit (%) of 1 − KM, so that the y axis shows the whole band."""
    return max((1 - km_at(k, tt)[1]) * 100 for k in kms for tt in k['t'])
def y_top(v):
    """Axis top and tick step: steps of 2 up to 10%, of 5 above."""
    step = 2 if v <= 10 else 5
    return int(np.ceil(v / step) * step), step
yE, sE = y_top(band_top(KMRC.values())); yF, sF = y_top(band_top([KMD, KMDL]))
km_panel(axE, [(RCL[g], TD5[TD5['rc'] == g], RC_COL[g]) for g in [0, 1, 2]], FU_NEW, TQ5, 'New aneurysm (%)', (0, yE),
         range(0, yE + 1, sE), band_alpha=0.08)
km_panel(axF, [('In-hospital Z < 2', TD, C_DER), ('Dilation only', TDL, NEJM['orange'])], FU_NEW, TQ5, 'New aneurysm (%)', (0, yF),
         range(0, yF + 1, sF))
fig.align_ylabels([axA, axC, axE]); fig.align_ylabels([axB, axD, axF])     # y-axis titles in two columns
for ax, letter in zip([axA, axB, axC, axD, axE, axF], 'ABCDEF'):
    letter_at(fig, ax, letter, pad_in=0.08)
save_checked(fig, 'Figure5_risk_factors_and_selective_echocardiography')

s1 = STRAT[1]
import ast
MISSED = ' and '.join(f"{v} {k.lower()}" for k, v in ast.literal_eval(s1['missed']).items())   # '3 small and 2 medium'
nD, kD = sum(n for _, n in RF_D.values()), sum(k for k, _ in RF_D.values())
nV, kV = sum(n for _, n in RF_V.values()), sum(k for k, _ in RF_V.values())
LEGENDS['Figure 5'] = (
    "Figure 5. Risk factors for an aneurysm after discharge and risk-based selection for the first post-discharge echocardiogram. "
    "A–D, aneurysm (maximal Z ≥ 2.5) at the first echocardiogram 1–30 days after discharge in children discharged with a maximal "
    f"in-hospital Z score below 2.5 (derivation n = {nfmt(nD)}, {kD} aneurysms; validation n = {nV}, {kV} aneurysms). A, by number of "
    "risk factors (age <12 months, IVIG resistance, in-hospital dilation only [Z 2 to <2.5], and discharge CRP >20 mg/L or white blood "
    "cell count >12 × 10⁹/L); blue, derivation; red, validation. B, by the exact in-hospital maximal Z score, in the "
    f"{int(GR['n'].sum())} derivation children whose in-hospital maximum was reported as an exact Z value; Cochran–Armitage test for "
    f"trend P = 0.020; Firth odds ratio per 0.5 Z, {gc['OR']:.2f} (95% CI, {gc['CI_low']:.2f}–{gc['CI_high']:.2f}), and "
    f"{ga['OR']:.2f} ({ga['CI_low']:.2f}–{ga['CI_high']:.2f}) adjusted for age <12 months and IVIG resistance. C, by age at admission "
    "(derivation), after an uncomplicated in-hospital course (in-hospital Z < 2) and in all children discharged with a maximal Z below "
    f"2.5; infants younger than 3 months: {AGE_KN[('<3 mo', MEAS_C[0][0])][0]}/{AGE_KN[('<3 mo', MEAS_C[0][0])][1]} and "
    f"{AGE_KN[('<3 mo', MEAS_C[1][0])][0]}/{AGE_KN[('<3 mo', MEAS_C[1][0])][1]}. Whiskers in A–C, Wilson 95% CIs; labels, children with "
    "an aneurysm/children. D, echocardiograms avoided and aneurysms detected (sensitivity) if the first post-discharge echocardiogram "
    "were limited to children meeting each rule (all children; at least 1 risk factor; at least 2 risk factors; or in-hospital dilation "
    "only or IVIG resistance, open diamond); whiskers, bootstrap 95% CIs (derivation) and the Wilson 95% CI of the detected "
    f"proportion (validation); labels, aneurysms detected/aneurysms. With the ≥1-risk-factor rule, {s1['tot'] - s1['det']} aneurysms "
    f"were missed in the derivation cohort ({MISSED}). E and F, cumulative "
    "incidence of a newly detected aneurysm after discharge (1 − Kaplan–Meier; derivation; echocardiograms to February 28, 2025; time 0 "
    "= discharge; an aneurysm is dated at the first echocardiogram that showed it; follow-up censored at the last echocardiogram and at "
    "730 days; shaded bands, 95% CIs). E, children with an uncomplicated in-hospital course and at least 1 echocardiogram after "
    f"discharge (n = {nfmt(len(TD5))}) by number of risk factors (age <12 months, IVIG resistance, and discharge CRP >20 mg/L or white "
    f"blood cell count >12 × 10⁹/L): at 1 year {RCL[0]} {ci_txt(KMRC[0], 365)}, {RCL[1]} {ci_txt(KMRC[1], 365)}, and {RCL[2]} "
    f"{ci_txt(KMRC[2], 365)}; log-rank {p_txt(lr_rc[2])}. F, children whose in-hospital maximum was dilation only (n = {len(TDL)}) "
    f"compared with an uncomplicated course (in-hospital Z < 2; n = {nfmt(len(TD))}): at 1 year {ci_txt(KMDL, 365)} versus {ci_txt(KMD, 365)}; log-rank "
    f"{p_txt(lr_dil[2])}. CRP, C-reactive protein; IVIG, intravenous immunoglobulin.")
print(LEGENDS['Figure 5'])

## 12. Supplemental figure — published cohorts beside this cohort (no pooling)
Published counts come from the extraction sheet `comparison/comparison_extraction_v2.csv` (each count checked against the
article text by `build_comparison_v2.py`); this cohort's counts come from the locks. Ma et al's late count is persistence of an
earlier finding, not a new finding, and Lee et al report no denominator; neither is plotted.

In [ ]:
from matplotlib import transforms as mtransforms
CMP = pd.read_csv(IN / 'comparison' / 'comparison_extraction_v2.csv')
def cmp_row(study, window_group, outcome_start):
    r = CMP[(CMP['study'] == study) & (CMP['window_group'] == window_group) & CMP['outcome'].str.startswith(outcome_start)]
    assert len(r) == 1, (study, window_group, outcome_start, len(r))
    return int(r['events'].iloc[0]), int(r['n'].iloc[0])
LAB = {'Ma 2019': 'Ma 2020', 'Tuohy 2001': 'McMorrow Tuohy 2001'}     # first author and year of the journal issue, as in the references
def pub(study, wg, oc, finding, kind):
    k, n = cmp_row(study, wg, oc)
    return dict(study=LAB.get(study, study), finding=finding, k=k, n=n, kind=kind, src='pub')
def own(label, finding, k, n, kind, src, lock=None):
    assert_lock(lock or (f'{k}/{nfmt(n)}' if n >= 1000 else f'{k}/{n}'))
    return dict(study=label, finding=finding, k=k, n=n, kind=kind, src=src)
GROUPS = [
    ('First echocardiogram after the acute assessment', [
        pub('Ma 2019', 'first follow-up', 'any involvement', 'Z ≥ 2', 'any'),
        pub('Ma 2019', 'first follow-up', 'aneurysm', 'Aneurysm', 'an'),
        pub('Wang 2022', 'first follow-up', 'any involvement', 'Z ≥ 2', 'any'),
        pub('Wang 2022', 'first follow-up', 'aneurysm', 'Aneurysm', 'an'),
        pub('Matsuoka 2020', 'first follow-up', 'aneurysm', 'Aneurysm', 'an'),
        pub('Tuohy 2001', 'first follow-up', 'new abnormality', 'Any abnormality', 'any'),
        pub('Tuohy 2001', 'first follow-up', 'new aneurysm', 'Aneurysm', 'an'),
        own('This study, derivation', 'Z ≥ 2', 35, 1618, 'any', 'der'),
        own('This study, derivation', 'Aneurysm', 26, 1618, 'an', 'der'),
        own('This study, derivation‡', 'Aneurysm', 33, 1702, 'an', 'der', lock='n = 1,702, 33 aneurysms'),
        own('This study, validation', 'Z ≥ 2', 11, 615, 'any', 'val'),
        own('This study, validation', 'Aneurysm', 6, 615, 'an', 'val')]),
    ('Second echocardiogram after normal earlier echocardiograms', [
        pub('de Ferranti 2018', 'second follow-up', 'any involvement', 'Z ≥ 2', 'any'),
        pub('de Ferranti 2018', 'second follow-up', 'aneurysm', 'Aneurysm', 'an'),
        pub('Wongbundit 2023', 'second follow-up', 'any involvement', 'Z ≥ 2', 'any'),
        own('This study, derivation', 'Z ≥ 2', 5, 1028, 'any', 'der'),
        own('This study, derivation', 'Aneurysm', 1, 1028, 'an', 'der'),
        own('This study, validation', 'Z ≥ 2', 2, 374, 'any', 'val'),
        own('This study, validation', 'Aneurysm', 0, 374, 'an', 'val')]),
    ('Later echocardiograms after normal earlier echocardiograms', [
        pub('Scott 1999', 'late', 'any abnormality', 'Any abnormality', 'any'),
        pub('Tuohy 2001', 'late', 'any abnormality', 'Any abnormality', 'any'),
        pub('Wang 2022', 'late', 'any involvement', 'Z ≥ 2', 'any'),
        pub('Wang 2022', 'late', 'aneurysm', 'Aneurysm', 'an'),
        pub('Matsuoka 2020', 'late', 'new aneurysm', 'Aneurysm', 'an'),
        pub('Wongbundit 2023', 'late', 'worsening', 'Z ≥ 2', 'any'),
        own('This study, derivation, 91–365 d', 'Z ≥ 2', 2, 609, 'any', 'der'),
        own('This study, derivation, 91–365 d', 'Aneurysm', 0, 609, 'an', 'der'),
        own('This study, derivation, >365 d', 'Z ≥ 2', 0, 221, 'any', 'der'),
        own('This study, validation, 91–365 d', 'Aneurysm', 0, 117, 'an', 'val')])]
SRC_COL = {'pub': INK, 'der': C_DER, 'val': C_VAL}
lines = []
for title, rows in GROUPS:
    lines.append(('h', title))
    lines += [('r', r) for r in rows]
nl = len(lines)
H = 0.235 * nl + 0.9
fig = plt.figure(figsize=(FIG_W, H))
ax = fig.add_axes([0.58, 0.55 / H, 0.40, (H - 0.75) / H])
bt = mtransforms.blended_transform_factory(fig.transFigure, ax.transData)
CMP_ROWS = []
for i, (typ, item) in enumerate(lines):
    y = nl - 1 - i
    if typ == 'h':
        ax.text(0.01, y, item, transform=bt, ha='left', va='center', fontsize=9, fontweight='bold', color=INK)
        continue
    r = item; pct = r['k'] / r['n'] * 100; lo, hi = wilson(r['k'], r['n']); col = SRC_COL[r['src']]
    ax.errorbar(pct, y, xerr=[[max(pct - lo, 0.0)], [max(hi - pct, 0.0)]], fmt='o', ms=5.2, mfc=col if r['kind'] == 'an' else WHITE, mec=col, ecolor=col,
                elinewidth=1.0, capsize=2, zorder=3)
    ax.text(0.03, y, r['study'], transform=bt, ha='left', va='center', fontsize=8.5, color=col)
    ax.text(0.37, y, r['finding'], transform=bt, ha='left', va='center', fontsize=8.5, color=col)
    ax.text(0.555, y, f"{r['k']}/{nfmt(r['n'])}", transform=bt, ha='right', va='center', fontsize=8.5, color=col)
    CMP_ROWS.append(dict(window=[g for g, rows in GROUPS if r in rows][0], study=r['study'], finding=r['finding'], events=r['k'], n=r['n'],
                         pct=round(pct, 2), ci_low=round(lo, 2), ci_high=round(hi, 2)))
ax.set_ylim(-0.7, nl - 0.3); ax.set_yticks([]); ax.spines['left'].set_visible(False)
ax.set_xlim(-0.45, 16); ax.set_xticks(range(0, 17, 4)); ax.set_xlabel('Children with the finding (%)')
from matplotlib.lines import Line2D
hd = [Line2D([], [], marker='o', ls='none', mfc=INK, mec=INK, label='Aneurysm'),
      Line2D([], [], marker='o', ls='none', mfc=WHITE, mec=INK, label='Z ≥ 2 or any abnormality')]
ax.legend(handles=hd, loc='lower right', fontsize=8.5, handlelength=1.0, borderaxespad=0.1)
FIG_S['published'] = 'FigureS_published_cohorts'
save_checked(fig, FIG_S['published'])
CMP_TABLE = pd.DataFrame(CMP_ROWS)
LEG_S['published'] = (
    "Findings at follow-up echocardiograms after normal earlier assessments: published cohorts beside this cohort. Estimates are "
    "shown side by side and were not pooled, because baseline definitions, Z-score systems and timing differ. Filled circles, "
    "aneurysm; open circles, Z ≥ 2 or any abnormality; whiskers, Wilson 95% CIs; black, published cohorts; blue, this study's derivation "
    "cohort; red, this study's validation cohort. Baseline, Z system and timing of each study: Ma 2020, Z < 2 in the left anterior "
    "descending and right coronary arteries on illness days 1–10, Boston Z scores, second echocardiogram on illness days 11–24; Wang "
    "2022, maximal Z < 2.5 within 9 days of treatment, Kobayashi (lambda-mu-sigma) Z scores, 1 month after treatment and then 2 months "
    "to 5 years after a normal 1-month echocardiogram; Matsuoka 2020, pretreatment maximal Z ≤ 2.0 in IVIG responders, Kobayashi Z "
    "scores, 4 weeks after onset and later to 24 months; McMorrow Tuohy 2001, normal initial echocardiogram (median illness day 6), no Z "
    "score (ectasia and aneurysm judged against normal values and adjacent segments), 1–2 months and more than 5 months after onset; de "
    "Ferranti 2018 (abstract), Z < 2 at baseline and 2 weeks, 6 (±3) weeks of illness; Wongbundit 2023, Z < 2 at baseline and 2 weeks, "
    "8 weeks and 1 year; Scott 1999, normal echocardiogram 2 weeks to 2 months after onset, later echocardiograms over a median 12.5 "
    "months. This study: Z < 2 on every in-hospital report, windows counted from discharge (first echocardiogram 1–30 days, then "
    "31–90, 91–365 and more than 365 days after normal earlier windows); ‡children discharged with a maximal Z below 2.5. Not plotted: "
    "Ma 2020 at the late visit (persistence of earlier findings, 5 of 164) and Lee 2005 (no new abnormality after normal echocardiograms "
    "at 1–2 months; denominator not reported in the abstract). IVIG, intravenous immunoglobulin.")
print(CMP_TABLE.to_string(index=False))

## 13. Supplemental tables
Each table is written as a CSV (UTF-8) for the supplement. Locked CSV tables are reformatted without changing values;
values taken from the locked Results text, the coronary-classification lock and the Z-equation check are asserted against
those files. Notes are stored without table numbers.

In [ ]:
TAB_S, TNOTE_S = {}, {}
COR_LOCK = open(COR / 'Results_locked_coronary_v1.txt', encoding='utf-8').read()
ZCHK = open(IN / 'methods_checks' / 'z_equation_source_check_v3_output_2026-10-25.txt', encoding='utf-8').read()
def in_text(text, *strings):
    for s in strings:
        assert s in text, f'"{s}" not found'
def pci(p, lo, hi, d=2):
    return f'{p:.{d}f} ({lo:.{d}f}–{hi:.{d}f})'
def kn(k, n):
    return f'{int(k)}/{nfmt(n)}'

# --- definitions -------------------------------------------------------------------------------------------------------
TAB_S['definitions'] = pd.DataFrame([
    ('Index admission', 'First inpatient admission with a discharge diagnosis of Kawasaki disease. Derivation: index admissions '
     '2018–2024 except 2021 (in-hospital echocardiography reports of 2021 were not in the export). Validation: index admissions whose '
     'first in-hospital echocardiogram was on or after March 8, 2025, the start of the structured measurement export.'),
    ('Coronary category of a report', 'AHA 2017 category of the largest Z score of the left main, left anterior descending and right '
     'coronary arteries (and the circumflex when written): no involvement, Z < 2; dilation only, 2 to < 2.5; small aneurysm, 2.5 to < 5; '
     'medium aneurysm, 5 to < 10; large or giant aneurysm, Z ≥ 10 or internal diameter ≥ 8 mm. Reports on the same day: the largest '
     'category (validation).'),
    ('Findings', 'Any involvement, Z ≥ 2; aneurysm, Z ≥ 2.5; medium or larger aneurysm, Z ≥ 5.'),
    ('Uncomplicated in-hospital course', 'Z < 2 on every classifiable echocardiography report of the index admission.'),
    ('Windows after discharge', 'Calendar days after the discharge date: 1–30, 31–90, 91–365 and more than 365 days.'),
    ('First post-discharge echocardiogram', 'The first classifiable report 1–30 days after discharge; its category is the outcome.'),
    ('Later windows', 'Children with Z < 2 on every report in all earlier windows and at least one report in the window; the outcome is '
     'the largest category in the window.'),
    ('Complete follow-up (validation)', 'Discharge at least 30, 90 or 365 days before the end of data (September 23, 2026) for the '
     '1–30-, 31–90- and 91–365-day windows.'),
    ('Numeric era (derivation)', 'Echocardiograms up to February 28, 2025. From March 2025 most measurements were held in a structured '
     'table that was exported separately and used only for the validation cohort.'),
    ('IVIG course', 'IVIG orders; an order starting 36 hours or more after the start of the previous course began a new course.'),
    ('IVIG resistance', 'A second IVIG course, a systemic glucocorticoid started more than 24 hours after the first IVIG, or '
     'infliximab during the index admission.'),
    ('Laboratory values', 'Admission value, first result of the index admission; discharge value, last result before discharge. CRP: '
     'standard CRP, or high-sensitivity CRP when standard CRP was not measured.'),
    ('Risk factors for selection', 'Age < 12 months; IVIG resistance; in-hospital dilation only (Z 2 to < 2.5); discharge CRP '
     '> 20 mg/L or white blood cell count > 12 × 10⁹/L (counted once).'),
    ('Readmission within 30 days', 'Any inpatient admission within 30 days after the index discharge.'),
    ('Illness day', 'Day 1 is the first day of fever, from the duration of fever recorded in the admission note.')],
    columns=['Term', 'Definition'])
TNOTE_S['definitions'] = ("Study definitions. AHA, American Heart Association; CRP, C-reactive protein; IVIG, intravenous "
                          "immunoglobulin.")

# --- Z equations -------------------------------------------------------------------------------------------------------
coef = {'LMCA': ('-0.3684', '4.9000', '-1.7633', '0.3242', '3,859'), 'LAD': ('-0.3806', '4.2197', '-1.5673', '0.2882', '3,722'),
        'RCA': ('-0.5758', '5.0284', '-2.1865', '0.3321', '4,881')}
for v, (a, b, c, sd, n) in coef.items():
    in_text(COR_LOCK, f'{v}: a {a}, b {b}, c {c}, SD {sd} mm (n = {n} pairs, 2018–2026)')
in_text(COR_LOCK, '12,492 diameter–Z pairs from 4,061 reports printing height and weight; Stevenson BSA',
        'median |difference| 0.003; within 0.1 in 94.7%, within 0.5 in 97.3%')
EQ = [('Laboratory equation (reconstructed), Stevenson BSA', 'lab equation (a + b sqrt(BSA) + c BSA, Stevenson)'),
      ('Beijing 2013, Stevenson BSA', 'Beijing 2013, Stevenson BSA'), ('Beijing 2013, Haycock BSA', 'Beijing 2013, Haycock BSA'),
      ('Beijing 2013, DuBois BSA', 'Beijing 2013, DuBois BSA'), ('Beijing 2013, Mosteller BSA', 'Beijing 2013, Mosteller BSA'),
      ('Shenzhen 2021, Stevenson BSA', 'Shenzhen 2021, Stevenson BSA'), ('Shenzhen 2021, Haycock BSA', 'Shenzhen 2021, Haycock BSA'),
      ('Dallaire 2011, Haycock BSA', 'Dallaire 2011, Haycock BSA'), ('Dallaire 2011, Stevenson BSA', 'Dallaire 2011, Stevenson BSA'),
      ('Dallaire 2011, DuBois BSA', 'Dallaire 2011, DuBois BSA'), ('Dallaire 2011, Mosteller BSA', 'Dallaire 2011, Mosteller BSA')]
sec1 = ZCHK.split('1. ALL MEASUREMENTS')[1].split('2. REPORTED')[0]
zrows = []
for lab, key in EQ:
    line = [l for l in sec1.splitlines() if re.match(r'\s*' + re.escape(key) + r'\s{2,}\d', l)]
    assert len(line) == 1, key
    vals = line[0][len(line[0].split(key)[0]) + len(key):].split()
    zrows.append(dict(Equation=lab, **{'Within 0.1, %': vals[0], 'Within 0.25, %': vals[1], 'Within 0.5, %': vals[2], 'Median |difference|': vals[3]}))
TAB_S['z_equations'] = pd.DataFrame(zrows)
sec6 = ZCHK.split('6. ILLUSTRATION')[1]
ill = {}
for v in ['LMCA', 'LAD', 'RCA']:
    m = re.search(v + r': lab equation ([\d.]+) / ([\d.]+) mm; Beijing ([\d.]+) / ([\d.]+) mm .*?Dallaire ([\d.]+) / ([\d.]+) mm', sec6)
    ill[v] = m.groups()
TAB_S['z_coefficients'] = pd.DataFrame([dict(Vessel=v, a=a, b=b, c=c, **{'SD, mm': sd, 'Pairs': n},
                                             **{'Diameter at Z 2.0/2.5, laboratory': f'{ill[v][0]}/{ill[v][1]}',
                                                'Beijing 2013': f'{ill[v][2]}/{ill[v][3]}', 'Dallaire 2011': f'{ill[v][4]}/{ill[v][5]}'})
                                        for v, (a, b, c, sd, n) in coef.items()])
TNOTE_S['z_equations'] = (
    "Coronary Z-score equation used by the hospital's reporting system and its agreement with published equations. Upper part: "
    "equation reconstructed from 12,492 pairs of a reported diameter and its reported Z score (2018–2026; 4,061 reports that printed "
    "height and weight): mean diameter = a + b × √BSA + c × BSA with a constant SD, BSA by the Stevenson formula; Z = (diameter − "
    "mean)/SD. Diameters (mm) at Z 2.0 and 2.5 are shown for a child of 100 cm and 16 kg. Lower part: percentage of the 12,492 "
    "reported Z scores reproduced within 0.1, 0.25 and 0.5 by each equation, and the median absolute difference. No published "
    "equation tested reproduced the reported values; the reconstructed equation reproduced them in every calendar year (88.0–97.5% "
    "within 0.1). [PLACEHOLDER: source of the laboratory equation — awaiting confirmation from the echocardiography laboratory.] "
    "BSA, body surface area; LAD, left anterior descending; LMCA, left main coronary artery; RCA, right coronary artery.")
in_text(ZCHK, '2020   631  88.0', '2025  1665  97.5')

# --- accuracy of the classification ------------------------------------------------------------------------------------
acc_rows = [
    ('Blinded extraction audit: all seven numeric fields agree', 'Development sample (200 reports)', 'First reading 186/200 (93.0%); after the corrections it prompted 195/200 (97.5%)'),
    ('', 'Validation sample (150 reports)', 'First reading 148/150 (98.7%); after the corrections it prompted 149/150 (99.3%)'),
    ('', 'Final check (100 reports)', 'First reading 95/100 (95.0%); after the corrections it prompted 99/100 (99.0%)'),
    ('Measurements from linked records (tier 3)', 'Same-date linkage (429 reports with their own numbers)', 'Exact category agreement 97.9%; aneurysm agreement 99.3%'),
    ('', 'Linkage within 7 days (487 reports)', 'Exact category agreement 98.6%; aneurysm agreement 99.8%'),
    ('Wording model (tier 4), cross-validated by child', 'All reports with a numeric category (n = 7,453)', 'Accuracy 83.7%; weighted κ 0.871; AUC for aneurysm 0.990; aneurysm yes/no accuracy 97.7%'),
    ('', 'Reports worded dilation or aneurysm (n = 2,884)', 'Accuracy 65.3%; weighted κ 0.586; AUC for aneurysm 0.853; aneurysm yes/no accuracy 96.0%'),
    ('', 'Reports worded normal (n = 4,569)', 'Accuracy 95.3%; weighted κ 0.752; aneurysm yes/no accuracy 98.8%'),
    ('', 'Wording category alone (benchmark)', 'Accuracy 78.0%; weighted κ 0.817; AUC for aneurysm 0.978'),
    ('', 'Temporal: trained to 2022, tested from 2023 (n = 2,803)', 'Accuracy 76.7%; weighted κ 0.835; AUC for aneurysm 0.986; aneurysm yes/no accuracy 97.1%'),
    ('Reports by evidence tier (all 30,003 reports)', 'Tier 1 written Z / tier 2 computed Z / tier 3 linked / tier 4 wording / none',
     '4,176 / 3,277 / 44 / 21,830 / 676'),
    ('Exact evidence among reports worded dilation or aneurysm', '2018–2024 (by year)', '96.3–100.0%'),
    ('', '2025 / 2026', '42.3% / 20.1%')]
for s in ['first look (development, before the fixes it prompted): all seven fields 186/200', 'first look (validation, before the fixes it prompted): all seven fields 148/150',
          'first look (final_check, before the fixes it prompted): all seven fields 95/100', 'all seven numeric fields agree 195/200 (97.5%)', 'all seven numeric fields agree 149/150 (99.3%)', 'all seven numeric fields agree 99/100 (99.0%)',
          'exact AHA agreement 97.9%, aneurysm agreement 99.3% (n = 429', 'exact AHA agreement 98.6%, aneurysm agreement 99.8% (n = 487',
          'n 7,453; exact 5-category accuracy 83.7%; weighted kappa 0.871; AUC aneurysm 0.990', 'aneurysm yes/no accuracy 97.7%',
          'n 2,884; exact 5-category accuracy 65.3%; weighted kappa 0.586; AUC aneurysm 0.853', 'aneurysm yes/no accuracy 96.0%',
          'n 4,569; exact 5-category accuracy 95.3%; weighted kappa 0.752', 'aneurysm yes/no accuracy 98.8%',
          'exact 5-category accuracy 78.0%; weighted kappa 0.817; AUC aneurysm 0.978',
          'n 2,803; exact 5-category accuracy 76.7%; weighted kappa 0.835; AUC aneurysm 0.986', 'aneurysm yes/no accuracy 97.1%',
          'tier 0: 676, tier 1: 4,176, tier 2: 3,277, tier 3: 44, tier 4: 21,830', '2020 96.3% of 188', '2025 42.3% of 626', '2026 20.1% of 407',
          'Echo reports 30,003']:
    in_text(COR_LOCK, s)
TAB_S['accuracy'] = pd.DataFrame(acc_rows, columns=['Check', 'Sample', 'Result'])
TNOTE_S['accuracy'] = (
    "Accuracy of the per-report coronary classification. Each audit prompted corrections of the extraction rules, so the values after "
    "correction were measured on the samples that prompted them and are optimistic. Tier 1, Z written by the sonographer; tier 2, Z computed from a written "
    "diameter with the laboratory equation; tier 3, measurements found in the discharge record or outpatient note for reports that "
    "referred to a table; tier 4, wording model (multinomial logistic regression on the report text, with an era term) for reports "
    "without measurements. Accuracy is over the five AHA categories. In the derivation analysis, 13 of 1,861 reports worded dilation or "
    "aneurysm relied on the wording model. AUC, area under the receiver operating characteristic curve.")
assert_lock('within it 13 of 1861 dilation/aneurysm-worded reports rely on the wording model')

# --- wording versus AHA category in hospital (cohort C1) ------------------------------------------------------------------
W = pd.read_csv(V41 / 'TableS4_inhospital_wording_vs_AHA.csv')
W = W[W['scope'] == 'primary'].drop(columns='scope').rename(columns={'inhospital_report_wording': 'Largest in-hospital wording',
                                                                       'None': 'No involvement', 'Large/giant': 'Large or giant'})
TAB_S['wording'] = W
assert_lock("wording 'dilation' (n = 389) → AHA: None 9, Dilation only 16, Small 266, Medium 82, Large/giant 16")
TNOTE_S['wording'] = ("Largest coronary wording of the in-hospital reports against the AHA category of the in-hospital maximum, "
                      "cohort C1 of the derivation cohort (n = 3,400). AHA, American Heart Association.")

# --- selection: children without an echocardiogram 1–30 days after discharge ----------------------------------------------
sel = [('Observed (children with the echocardiogram)', '26/1,618', '1.61 (1.10–2.34)'),
       ('Inverse probability weighting', '', '1.54 (0.98–2.21)'),
       ('Standardized to the risk-factor distribution of all 2,839 children', '', '1.55 (1.00–2.19)'),
       ('Lower bound: no aneurysm in any child without the echocardiogram', '26/2,839', '0.92 (0.63–1.34)'),
       ('Rate among children without the echocardiogram that would double the overall rate', '', '5.34'),
       ('Children without it who had an echocardiogram 31–365 days after discharge', '3/199', '1.51 (0.51–4.34)')]
assert_lock('observed 26/1,618 = 1.61% (95% CI 1.10–2.34); IPW 1.54% (bootstrap 95% CI 0.98–2.21); standardized to the risk-factor '
            'distribution 1.55% (1.00–2.19); lower bound 26/2,839 = 0.92% (95% CI 0.63–1.34); tipping point', '5.34%',
            'aneurysm at that echo 3/199 = 1.51% (95% CI 0.51–4.34)', 'propensity model c-statistic 0.632')
TAB_S['selection'] = pd.DataFrame(sel, columns=['Estimate', 'Aneurysms/children', '% (95% CI)'])
TNOTE_S['selection'] = (
    "Aneurysm at the first echocardiogram after discharge, allowing for the 1,221 derivation children with an uncomplicated "
    "in-hospital course who had no echocardiogram 1–30 days after discharge in the hospital's records (1,618 of 2,839 had one, "
    "57.0%; validation 615 of 860, 71.5%). The weights came from a logistic model for having the echocardiogram (c statistic 0.632); "
    "95% CIs for the weighted and standardized estimates are bootstrap intervals (2,000 resamples).")
assert_lock('615 (71.5%; v4 1,618/2,839 = 57.0%)')

# --- sensitivity analyses of the first-echo yield --------------------------------------------------------------------------
S2T = pd.read_csv(V41 / 'TableS2_sensitivity.csv')
M1A = pd.read_csv(V41 / 'TableM1a_2021_sensitivity.csv')
V52 = pd.read_csv(V5 / 'TableV5_2_P1_sensitivity.csv')
NAME_S2 = {0: 'Primary analysis', 1: 'Baseline: last in-hospital report Z < 2 (earlier involvement allowed)', 2: 'First post-discharge echocardiogram within 60 days',
           3: 'IVIG-treated children only', 4: 'Readmissions within 30 days excluded', 5: 'Index admissions 2018–2020',
           6: 'Index admissions 2022–2024', 7: 'First echocardiogram on illness day 11–21',
           8: 'Illness day 22–56, after a normal echocardiogram on day 11–21', 9: 'Events only from reports with exact numbers (tiers 1–3)',
           10: 'All index years 2018–2026 and all echocardiograms (wording model for reports without numbers)'}
rows = []
for i, r in S2T.iterrows():
    assert_lock(f"{int(r['events'])}/{int(r['n'])} ({r['pct']:.2f}%)")
    rows.append(dict(Cohort='Derivation', Analysis=NAME_S2[i], **{'Aneurysms/children': kn(r['events'], r['n']), '% (95% CI)': pci(r['pct'], r['ci_low'], r['ci_high']),
                     'Any involvement, n': '' if pd.isna(r['involvement']) else str(int(r['involvement']))}))
for _, r in M1A[M1A['outcome'].str.startswith('aneurysm') & ~M1A['analysis'].str.startswith('Primary')].iterrows():
    inv = M1A[M1A['outcome'].str.startswith('any') & (M1A['analysis'] == r['analysis'])]['events'].iloc[0]
    lab = {'Rule-based baseline, 2018–2024 without 2021 (cohort C1; agreement check)': 'Rule-based in-hospital baseline, 2018–2024 without 2021 (check)',
           'Rule-based baseline, 2021 only': 'Rule-based in-hospital baseline, 2021 only',
           'Pooled 2018–2024 including 2021 (AHA baseline + 2021 rule-based)': 'Pooled 2018–2024 including 2021'}[r['analysis']]
    rows.append(dict(Cohort='Derivation', Analysis=lab, **{'Aneurysms/children': kn(r['events'], r['n']), '% (95% CI)': pci(r['pct'], r['ci_low'], r['ci_high']),
                     'Any involvement, n': str(int(inv))}))
for _, r in V52.iterrows():
    lab = {'Primary: combined rule': 'Primary analysis (structured and text numbers, wording override)',
           '(a) without the wording override': 'Without the wording override',
           '(b) v4 report categories (wording model in this period)': 'Derivation-era report categories (wording model)'}[r['analysis']]
    rows.append(dict(Cohort='Validation', Analysis=f"{lab}; predictive P = {r['predictive_P']:.2f}",
                     **{'Aneurysms/children': kn(r['events'], r['n']), '% (95% CI)': pci(r['pct'], r['ci_low'], r['ci_high']), 'Any involvement, n': ''}))
TAB_S['sensitivity'] = pd.DataFrame(rows)
assert_lock('Rule-based baseline, 2021 only 7/515 = 1.36% (95% CI 0.66–2.78)', '(a) without the wording override: 5/615 = 0.81% (95% CI 0.35–1.89)')
TNOTE_S['sensitivity'] = (
    "Sensitivity analyses of the aneurysm yield (Z ≥ 2.5) at the first echocardiogram 1–30 days after an uncomplicated in-hospital "
    "course. The 2021 rows use a rule-based in-hospital baseline from the discharge record because the 2021 in-hospital reports were "
    "not exported; its agreement check in 2018–2024 is shown. Validation P values: beta-binomial predictive test against the "
    "derivation estimate. Intervals: Wilson 95% CIs. IVIG, intravenous immunoglobulin.")

# --- infants ---------------------------------------------------------------------------------------------------------------
INF2 = pd.read_csv(V41 / 'TableM1b_infants.csv').rename(columns={
    'age_group': 'Age', 'C1_n': 'Cohort C1, n', 'inhosp_aneurysm': 'In-hospital aneurysm', 'inhosp_medium_plus': 'In-hospital medium or larger',
    'aneurysm_first_after_discharge_of_all_aneurysms': 'Aneurysms first seen after discharge/all aneurysms',
    'first_echo_aneurysm_uncomplicated': 'First post-discharge echo: aneurysm after an uncomplicated course',
    'first_echo_aneurysm_Z_lt_2_5': 'First post-discharge echo: aneurysm after discharge with Z < 2.5'})
TAB_S['infants'] = INF2
IF = pd.read_csv(V41 / 'TableM1b_infant_firth.csv')
TAB_S['infants_firth'] = pd.DataFrame([dict(Outcome=r['outcome'].replace('in-hospital', 'In-hospital'), Variable=r['variable'],
                                            **{'Odds ratio (95% CI)': f"{r['OR']:.2f} ({r['CI_low']:.2f}–{r['CI_high']:.2f})",
                                               'P value': ('<0.001' if r['p'] < 0.001 else f"{r['p']:.3f}" if r['p'] < 0.1 else f"{r['p']:.2f}")})
                                       for _, r in IF.iterrows()])
assert_lock('Age <6 months (vs ≥12) OR 3.19 (2.33–4.38)', 'Age <6 months (vs ≥12) OR 3.29 (2.09–5.18)')
TNOTE_S['infants'] = ("Infants in the derivation cohort. Cells give children with the finding/children (%; Wilson 95% CI). Lower part: "
                      "Firth logistic regression for the in-hospital findings in cohort C1 (n = 3,400; 432 children with an aneurysm and "
                      "152 with a medium or larger aneurysm), with age, sex, and IVIG resistance in the model. Echo, echocardiogram; IVIG, "
                      "intravenous immunoglobulin.")

# --- measurement noise -----------------------------------------------------------------------------------------------------
rp_rows = []
for _, r in RP.iterrows():
    rp_rows.append(dict(Pairs=r['set'].replace('A: index admission, ≤72 h apart', 'In hospital, 1–3 days apart').replace(
        'B: uncomplicated, >30 d after discharge, ≤180 d apart', 'Uncomplicated course, >30 days after discharge, ≤180 days apart'),
        **{'Pairs (children)': f"{int(r['pairs'])} ({int(r['children'])})", 'Median interval, d': f"{r['median_interval_days']:.0f}",
           'Mean difference': f"{r['mean_diff']:.2f}", 'SD of differences': f"{r['sd_diff']:.2f}", 'Within-child SD': f"{r['within_sd']:.2f}",
           'Repeatability coefficient': f"{r['repeatability_coef']:.2f}", '|Difference| > 2, %': f"{r['pct_abs_diff_gt_2']:.1f}",
           'Robust within-child SD*': f"{r['posthoc_robust_within_sd']:.2f}", 'Robust repeatability coefficient*': f"{r['posthoc_robust_repeatability_coef']:.2f}"}))
assert_lock('287 pairs (220 children; median interval 3.0 d)', '189 pairs (109 children; median interval 73.0 d)')
TAB_S['noise'] = pd.DataFrame(rp_rows)
TNOTE_S['noise'] = ("Repeatability of the maximal coronary Z score between consecutive reports with exact numbers. Repeatability "
                    "coefficient, 2.77 × within-child SD. *Added after the module plan was locked, because the differences were "
                    "heavy-tailed: within-child SD from the median absolute deviation of the differences. The event definitions "
                    "and the noise model built on these estimates are shown in Figure 3.")

# --- Z gradient below the aneurysm threshold ----------------------------------------------------------------------------------
G = pd.read_csv(V41 / 'TableM3a_Z_gradient.csv'); GF = pd.read_csv(V41 / 'TableM3a_Z_gradient_firth.csv')
for _, r in G.iterrows():
    assert_lock(f"{int(r['events'])}/{int(r['n'])} ({r['pct']:.2f}%; {r['ci_low']:.2f}–{r['ci_high']:.2f})")
grad = [dict(Group=f"Exact in-hospital maximal Z {r['in_hospital_exact_max_Z'].replace('–<', ' to <')}", **{'Aneurysms/children': kn(r['events'], r['n']),
             '% (95% CI) or OR (95% CI)': pci(r['pct'], r['ci_low'], r['ci_high'])}) for _, r in G.iterrows()]
lab_gf = {'z_per05': 'Per 0.5 increase in Z', 'lt12': 'Age <12 months', 'ivr': 'IVIG resistance'}
for _, r in GF.iterrows():
    grad.append(dict(Group=f"Firth OR, {r['model']}: {lab_gf[r['term']]}", **{'Aneurysms/children': '',
                     '% (95% CI) or OR (95% CI)': f"{r['OR']:.2f} ({r['CI_low']:.2f}–{r['CI_high']:.2f}); P = {r['p']:.3f}"}))
assert_lock('crude 1.39 (1.06–1.84), P = 0.019; adjusted for age <12 mo and IVIG resistance 1.34 (1.02–1.76), P = 0.036', 'Cochran–Armitage trend P = 0.020')
TAB_S['z_gradient'] = pd.DataFrame(grad)
TNOTE_S['z_gradient'] = ("Aneurysm at the first echocardiogram after discharge by the exact in-hospital maximal Z score, among the 358 of "
                         "1,702 children discharged with Z below 2.5 whose in-hospital maximum was an exact value (15 aneurysms). "
                         "Cochran–Armitage test for trend P = 0.020. OR, odds ratio; IVIG, intravenous immunoglobulin.")

# --- timing of the first post-discharge echocardiogram --------------------------------------------------------------------------
TM = pd.read_csv(V41 / 'TableM3c_timing.csv'); TF = pd.read_csv(V41 / 'TableM3c_timing_firth.csv')
tim = []
for _, r in TM.iterrows():
    assert_lock(f"aneurysm {int(r['aneurysm_events'])}/{int(r['aneurysm_n'])} ({r['aneurysm_pct']:.2f}%)")
    tim.append(dict(Timing=f"{'Days after discharge' if r['variable'].startswith('days') else 'Illness day'} {r['level'].replace(' d', '')}",
                    Aneurysm=f"{kn(r['aneurysm_events'], r['aneurysm_n'])}; {pci(r['aneurysm_pct'], r['aneurysm_ci_low'], r['aneurysm_ci_high'])}",
                    **{'Any involvement': f"{kn(r['involvement_events'], r['involvement_n'])}; {pci(r['involvement_pct'], r['involvement_ci_low'], r['involvement_ci_high'])}"}))
tl = {'d_1–7 d': 'Days 1–7 vs 8–14 after discharge', 'd_15–30 d': 'Days 15–30 vs 8–14 after discharge', 'i_≤14': 'Illness day ≤14 vs 15–21',
      'i_22–30': 'Illness day 22–30 vs 15–21', 'i_>30': 'Illness day >30 vs 15–21'}
for _, r in TF[TF['term'].isin(tl)].iterrows():
    tim.append(dict(Timing=f"Firth OR adjusted for the risk-factor count: {tl[r['term']]}", Aneurysm=f"{r['OR']:.2f} ({r['CI_low']:.2f}–{r['CI_high']:.2f})",
                    **{'Any involvement': ''}))
TAB_S['timing'] = pd.DataFrame(tim)
TNOTE_S['timing'] = ("Findings at the first echocardiogram after an uncomplicated in-hospital course by its timing (derivation, "
                     "n = 1,618; illness day available for 1,253). Cells: children with the finding/children; % (Wilson 95% CI). "
                     "OR, odds ratio.")

# --- risk-based selection strategies ------------------------------------------------------------------------------------------
st_rows = []
SNAME = {'S0': 'Echocardiogram for all', 'S1': '≥1 risk factor', 'S2': '≥2 risk factors', 'S3': 'Dilation only or IVIG resistance',
         'S4': '≥1 risk factor or exact in-hospital Z ≥ 1.5 (exploratory)'}
for _, r in M3B.iterrows():
    st_rows.append(dict(Cohort='Derivation', Strategy=SNAME[r['strategy'][:2]], **{
        'Echocardiograms': nfmt(r['echoes']), 'Avoided, % (95% CI)': f"{float(r['echoes_avoided_pct']):.1f} ({r['echoes_avoided_pct_ci']})",
        'Detected/aneurysms': f"{int(r['detected'])}/{int(r['aneurysms'])}", 'Sensitivity, % (95% CI)': f"{float(r['sensitivity_pct']):.1f} ({r['sensitivity_pct_ci']})",
        'Missed, by size': '' if r['missed_by_size'] in ('{}', '') or pd.isna(r['missed_by_size']) else r['missed_by_size'].strip('{}').replace("'", '').replace(':', ''),
        'Echocardiograms per aneurysm detected': f"{float(r['echoes_per_aneurysm_detected']):.1f}"}))
s7 = KN5['S7']
st_rows.append(dict(Cohort='Validation', Strategy='≥1 risk factor', **{
    'Echocardiograms': nfmt(s7['echoes_if_ge1']), 'Avoided, % (95% CI)': f"{s7['echoes_avoided_pct']:.1f}",
    'Detected/aneurysms': f"{s7['detected']}/{s7['aneurysms']}", 'Sensitivity, % (95% CI)': f"{s7['sensitivity']['pct']:.1f} ({s7['sensitivity']['ci_low']:.1f}–{s7['sensitivity']['ci_high']:.1f})",
    'Missed, by size': '', 'Echocardiograms per aneurysm detected': f"{s7['echoes_if_ge1'] / s7['detected']:.1f}"}))
assert_lock(f"echoes {s7['echoes_if_ge1']} (avoided {s7['echoes_avoided_pct']:.1f}%); detected {s7['detected']}, missed 0")
TAB_S['strategies'] = pd.DataFrame(st_rows)
TNOTE_S['strategies'] = (
    "Echocardiograms avoided and aneurysms detected if the first echocardiogram after discharge were limited to children meeting "
    "each rule; children discharged with an in-hospital maximal Z below 2.5 and an echocardiogram 1–30 days after discharge "
    f"(derivation n = 1,702, 33 aneurysms; validation n = {s7['children']}, {s7['aneurysms']} aneurysms). Risk factors: age <12 months, "
    "IVIG resistance, in-hospital dilation only (Z 2 to <2.5), discharge CRP >20 mg/L or white blood cell count >12 × 10⁹/L. "
    "Derivation 95% CIs: bootstrap (2,000 resamples); validation sensitivity: Wilson 95% CI. CRP, C-reactive protein; IVIG, "
    "intravenous immunoglobulin.")

# --- management after a post-discharge aneurysm --------------------------------------------------------------------------------
MG = pd.read_csv(V41 / 'TableM4a_after_postdischarge_aneurysm.csv')
MG['group'] = ['Aneurysm first documented after discharge (cohort C1)', 'Found at the first echocardiogram after an uncomplicated course',
               'Comparator: small aneurysm in hospital (time 0 = discharge)']
TAB_S['management'] = MG.rename(columns={'group': 'Group', 'n': 'n', 'reimaged_90d': 'Re-imaged ≤90 d', 'reimaged_365d': 'Re-imaged ≤365 d*',
                                         'median_days_to_next_echo': 'Median days to next echo', 'inpatient_admission_365d': 'Admitted ≤365 d*',
                                         'inpatient_antithrombotic_order_365d': 'Inpatient antithrombotic order ≤365 d*',
                                         'inpatient_order_beyond_aspirin_365d': 'Beyond aspirin*', 'outpatient_note_90d_available': 'Outpatient note ≤90 d†',
                                         'note_mentions_antithrombotic': 'Note mentions antithrombotic', 'note_mentions_beyond_aspirin': 'Note mentions beyond aspirin',
                                         'note_mentions_aneurysm': 'Note mentions aneurysm'})
assert_lock('re-imaged ≤90 d 22/26, ≤365 d 19/22; median days to the next echo 14.0', '365 d 86.34% (72.17–95.49; n = 40, 31 events)',
            'ordered in the index admission 199/280')
TNOTE_S['management'] = (
    "Care after an aneurysm first documented after discharge (descriptive; derivation cohort). Inpatient orders cover hospital "
    "admissions only; outpatient notes were available from January 3, 2021. *Denominator: children with at least 365 days of "
    "potential follow-up. †Denominator: children with a visit note available. After a post-discharge aneurysm, the cumulative "
    "probability of a report below Z 2.5 was 60.27% by 90 days and 86.34% by 365 days (Kaplan–Meier, n = 40). For reference, "
    "199 of 280 children with a small in-hospital aneurysm had an antithrombotic beyond aspirin ordered during the index admission. "
    "Echo, echocardiogram.")
assert_lock('90 d 60.27% (45.18–75.77; n = 40, 31 events)')

# --- validation endpoints ------------------------------------------------------------------------------------------------------
EPN = {'P1': 'P1. Aneurysm, first echocardiogram 1–30 d', 'S1': 'S1. Any involvement, first echocardiogram 1–30 d',
       'S2': 'S2. Medium or larger, first echocardiogram 1–30 d', 'S4_aneurysm': 'S4. Aneurysm, 31–90 d after normal 1–30-d echocardiograms',
       'S4_involvement': 'S4. Any involvement, 31–90 d (comparison not prespecified)'}
ve = []
for key, r in V51.iterrows():
    ve.append(dict(Endpoint=EPN[key], Validation=f"{kn(r['v5_events'], r['v5_n'])}; {pci(r['v5_pct'], r['v5_ci_low'], r['v5_ci_high'])}",
                   Derivation=f"{kn(r['v4_events'], r['v4_n'])}; {pci(r['v4_pct'], r['v4_ci_low'], r['v4_ci_high'])}",
                   **{'Difference, percentage points (95% CI)': f"{r['diff_pp']:+.2f} ({r['diff_ci_low']:+.2f} to {r['diff_ci_high']:+.2f})",
                      'Expected events': f"{r['expected_v5_events_under_v4']:.1f}", 'Predictive P': f"{r['predictive_P']:.3f}" if r['predictive_P'] < 0.1 else f"{r['predictive_P']:.2f}",
                      'Pooled 2018–2026': f"{kn(r['pooled_events'], r['pooled_n'])}; {pci(r['pooled_pct'], r['pooled_ci_low'], r['pooled_ci_high'])}"}))
for lab, val in [('S3. Aneurysm on any echocardiogram within 30 d', '6/615; 0.98 (0.45–2.11)'), ('S5. Aneurysm, 91–365 d after normal earlier windows', '0/117; 0.00 (0.00–3.18)'),
                 ('S6. P1 aneurysms confirmed on the next echocardiogram within 90 d', '6/6')]:
    ve.append(dict(Endpoint=lab, Validation=val, Derivation='', **{'Difference, percentage points (95% CI)': '', 'Expected events': '', 'Predictive P': '', 'Pooled 2018–2026': ''}))
assert_lock('S3 aneurysm on any echocardiogram within 30 d: 6/615 = 0.98% (95% CI 0.45–2.11)', 'aneurysm 0/117 = 0.00% (95% CI 0.00–3.18)', 'confirmed 6, not confirmed 0')
TAB_S['validation'] = pd.DataFrame(ve)
TNOTE_S['validation'] = (
    "Prespecified endpoints of the temporal validation (plan locked October 14, 2026 before any validation outcome was computed). "
    "Cells: children with the finding/children; % (Wilson 95% CI). Difference, validation minus derivation (Newcombe hybrid score "
    "interval). Expected events, validation events expected at the derivation rate. Predictive P, two-sided beta-binomial "
    "predictive test of the validation count against the derivation estimate; P ≥ 0.05 was read as no detectable incompatibility. "
    "S7 (risk-factor selection) is shown with the selection strategies, and S8 (91–365 and >365 days with follow-up extended to "
    "the end of the structured export) gave 0/654 and 0/328 aneurysms.")
assert_lock('W3 aneurysm 0/654 (95% CI 0.00–0.58)', 'W4 aneurysm 0/328 (95% CI 0.00–1.16)')

# --- validation events one by one ------------------------------------------------------------------------------------------------
V54b = V54.drop(columns=[c for c in ['research_patient_id', 'inhospital_report_days', 'later_echo_days'] if c in V54.columns]).copy()
V54b.insert(0, 'Event', range(1, len(V54b) + 1))
SRC_LAB = {'numbers': 'Structured measurements', 'numbers+wording_override': 'Structured measurements and report conclusion'}
assert set(V54b['source']) <= set(SRC_LAB), set(V54b['source'])
V54b['source'] = V54b['source'].map(SRC_LAB)
TAB_S['validation_events'] = V54b.rename(columns={'index_year': 'Index year', 'inhospital_max_structured_Z': 'In-hospital maximal Z (structured)',
                                                  'day_after_discharge': 'Day after discharge', 'category': 'Category', 'structured_Z': 'Z (structured)',
                                                  'text_Z': 'Z (text)', 'source': 'Source', 'status_next_90d': 'Next echo ≤90 d',
                                                  'next_echo_days_later': 'Days to next echo', 'next_echo_category': 'Category at next echo',
                                                  'last_echo_day_after_discharge': 'Last echo, day after discharge', 'last_echo_category': 'Category at last echo'})
TNOTE_S['validation_events'] = ("The 6 aneurysms at the first echocardiogram after an uncomplicated in-hospital course in the validation "
                                "cohort. Source: structured measurements, proximal segment measurements of the structured table; structured "
                                "measurements and report conclusion, a non-proximal segment also named as dilated in the report conclusion "
                                "(conclusion rule). Echo, echocardiogram. Research identifiers are not shown.")
assert_lock('days after discharge 8, 4, 3, 12, 10, 7')

# --- published cohorts (data of the comparison figure) ----------------------------------------------------------------------------
TAB_S['published'] = CMP_TABLE.rename(columns={'window': 'Window', 'study': 'Study', 'finding': 'Finding', 'events': 'Events', 'n': 'Children',
                                               'pct': '%', 'ci_low': '95% CI, low', 'ci_high': '95% CI, high'})
TNOTE_S['published'] = ("Counts behind the comparison figure. Published counts were extracted from the full text (abstract only for "
                        "de Ferranti 2018) and checked against the article text; 95% CIs are Wilson intervals computed here.")

# --- primary analysis versus all data (locked Table 7) ---------------------------------------------------------------------
T7L = {0: 'In-hospital aneurysm (Z ≥ 2.5), children with a classifiable in-hospital echocardiogram', 1: 'Aneurysm at the first echocardiogram 1–30 d after an uncomplicated course',
       2: 'Any involvement at the first echocardiogram 1–30 d after an uncomplicated course', 3: 'Any involvement at 31–90 d after normal 1–30-d echocardiograms',
       4: 'Aneurysm at the first echocardiogram 1–30 d after in-hospital dilation only', 5: 'Aneurysms first documented after discharge, % of all aneurysms',
       6: 'Small aneurysm: below Z 2.5 within 1 year', 7: 'Medium aneurysm: below Z 2.5 within 1 year', 8: 'Large or giant aneurysm: below Z 2.5 within 1 year',
       9: 'Small aneurysm: larger category within 1 year', 10: 'Medium aneurysm: larger category within 1 year',
       11: 'New aneurysm within 1 year after an uncomplicated course (cumulative incidence)'}
T7 = pd.read_csv(V41 / 'Table7_planB_primary_vs_alldata.csv')
assert len(T7) == len(T7L)
pb = []
for i, r in T7.iterrows():
    assert_lock(f"primary {r['value_primary']:.2f} ({r['ci_low_primary']:.2f}–{r['ci_high_primary']:.2f}; n = {int(r['n_primary'])}) | all data "
                f"{r['value_all']:.2f} ({r['ci_low_all']:.2f}–{r['ci_high_all']:.2f}; n = {int(r['n_all'])})")
    pb.append({'Estimate': T7L[i], 'Primary analysis, % (95% CI)': pci(r['value_primary'], r['ci_low_primary'], r['ci_high_primary']),
               'Primary, n': nfmt(r['n_primary']), 'All data, % (95% CI)': pci(r['value_all'], r['ci_low_all'], r['ci_high_all']), 'All data, n': nfmt(r['n_all'])})
TAB_S['plan_b'] = pd.DataFrame(pb)
TNOTE_S['plan_b'] = ("Estimates from the primary analysis (index admissions 2018–2024 without 2021; echocardiograms to February 28, 2025, when nearly "
                     "every abnormal report carried measurements) compared with all available data (index admissions 2018–2026 without 2021 and all "
                     "echocardiograms; the wording model supplied the category of reports without measurements, most of them from March 2025). "
                     "Proportions with Wilson 95% CIs; Kaplan–Meier estimates (time 0 = discharge) with log-log 95% CIs.")

for k, df in TAB_S.items():
    df.to_csv(OUT / f'TableS_{k}.csv', index=False, encoding='utf-8-sig')
print('supplemental tables:', {k: df.shape for k, df in TAB_S.items()})

## 14. Legends, notes, descriptive numbers and manifest

In [ ]:
MAIN_FIGS = ['Figure 1', 'Figure 2', 'Figure 3', 'Figure 4', 'Figure 5']
assert all(k in LEGENDS for k in MAIN_FIGS) and all(k in TABLE_NOTES for k in ['Table 1', 'Table 2'])
with open(OUT / 'Figure_legends_and_table_notes_main.txt', 'w', encoding='utf-8') as fh:
    fh.write('FIGURE LEGENDS AND TABLE NOTES (JAHA display items; numbers inserted from the locked outputs)\n\n')
    for k in MAIN_FIGS:
        fh.write(LEGENDS[k] + '\n\n')
    for k in ['Table 1', 'Table 2']:
        fh.write(TABLE_NOTES[k] + '\n\n')
    fh.write(f'Font: {FONT}. Figures at most {MAX_W:.1f} in (178 mm) wide and {MAX_H:.1f} in high; PDF vector, TIFF and PNG at 600 dpi.\n')
with open(OUT / 'Tables_preview_main.txt', 'w', encoding='utf-8') as fh:
    fh.write(TABLE_NOTES['Table 1'] + '\n\n' + show.to_string(index=False) + '\n\n\n')
    fh.write(TABLE_NOTES['Table 2'] + '\n\n' + TABLE2.to_string(index=False) + '\n')
with open(OUT / 'Supplement_legends_and_notes.json', 'w', encoding='utf-8') as fh:
    json.dump(dict(figures={k: dict(file=FIG_S[k], legend=LEG_S[k]) for k in LEG_S},
                   tables={k: dict(file=f'TableS_{k}.csv', note=TNOTE_S.get(k, '')) for k in TAB_S}), fh, ensure_ascii=False, indent=1)
with open(OUT / 'Descriptive_numbers_v3.json', 'w', encoding='utf-8') as fh:     # descriptive numbers quoted in the manuscript
    json.dump(EXTRA, fh, ensure_ascii=False, indent=1)
SIZES = pd.DataFrame([dict(figure=k, width_in=round(float(w), 2), height_in=round(float(h), 2)) for k, (w, h) in FIG_SIZE.items()])
SIZES.to_csv(OUT / 'Figure_sizes_v3.csv', index=False)                          # saved size of each figure (layout, not data)
print(SIZES.to_string(index=False))
assert (SIZES['width_in'] <= MAX_W).all() and (SIZES['height_in'] <= MAX_H).all()
man = [{'file': p.name, 'bytes': p.stat().st_size, 'sha256': sha256(p)} for p in sorted(OUT.iterdir()) if p.is_file() and p.name != 'MANIFEST_sha256.csv']
pd.DataFrame(man).to_csv(OUT / 'MANIFEST_sha256.csv', index=False)
print(len(man), 'files in', OUT.name)
print(pd.DataFrame(man)[['file', 'bytes']].to_string(index=False))

## 15. Self-check of the outputs
The tables, legends, notes and descriptive numbers (CSV, TXT, JSON) should match the reference run byte for byte
(`EXPECTED_OUTPUTS_sha256.csv`). Figures are not compared: their PNG/TIFF/PDF files carry the local font (Arial) and software
versions, which change the bytes but not the content; for the same reason `Figure_sizes_v3.csv` is not compared.

In [ ]:
EXP = BASE / 'EXPECTED_OUTPUTS_sha256.csv'
if EXP.exists():
    exp = pd.read_csv(EXP)
    res = [(f, (OUT / f).exists() and sha256(OUT / f) == h) for f, h in zip(exp['file'], exp['sha256'])]
    bad = [f for f, ok in res if not ok]
    print(f'{len(res) - len(bad)} of {len(res)} tables, legends and notes identical to the reference run' + ('' if not bad else '; different: ' + ', '.join(bad)))